# Setup

## 1. Environment Setup

We use [`uv`](https://github.com/astral-sh/uv) for fast, reproducible package management.

The steps below:
1. Install `uv` into `~/.local/bin`
2. Create a virtual environment at `.venv/`
3. Install all required packages (This might take a while)

> **After running this cell, restart the kernel** so that the newly installed packages (especially `vllm` and `transformers`) are picked up by the current Python session.

### IN Colab

In [1]:
#!unzip /content/repo.zip -d /content/CSE151B_Kaggle
%cd /content/CSE151B_Kaggle/CSE151B_Kaggle
!pwd
!ls

/content/CSE151B_Kaggle/CSE151B_Kaggle
/content/CSE151B_Kaggle/CSE151B_Kaggle
baseline			  Runner.html
data				  Runner.ipynb
prompting			  Runner.pdf
results				  run_prompt_strategy_experiments.py
run_baseline2.py		  run_ved_category_experiments.py
run_baseline3.py		  sample_problem.json
run_geometry_trig_experiments.py  tests


In [ ]:
!pip install prettyprint sympy numpy pandas matplotlib transformers accelerate tqdm bitsandbytes ipykernel jupyter nvidia-nvjitlink antlr4-python3-runtime==4.11.1

!pip install -U uv
!uv pip install --system --reinstall --torch-backend=cu128 "vllm==0.18"

In [ ]:
# !pip uninstall -y vllm vllm-flash-attn vllm-triton-backend
# !pip install -U uv
# !uv pip install --system --reinstall --torch-backend=cu128 "vllm==0.18"

In [2]:
import os

# Point Colab's environment to the newly installed nvidia-nvjitlink package
lib_path = "/usr/local/lib/python3.12/dist-packages/nvidia/nvjitlink/lib"

if "LD_LIBRARY_PATH" in os.environ:
    os.environ["LD_LIBRARY_PATH"] += f":{lib_path}"
else:
    os.environ["LD_LIBRARY_PATH"] = lib_path

In [3]:
import torch
print("torch:", torch.__version__)
print("torch cuda:", torch.version.cuda)
#print("cuda available:", torch.cuda.is_available())

import vllm
print("vLLM version:", vllm.__version__)

from vllm import LLM, SamplingParams
print("vLLM import OK")

torch: 2.10.0+cu128
torch cuda: 12.8
vLLM version: 0.18.0
vLLM import OK


In [ ]:
!zip -qr /content/CSE151B_Kaggle.zip /content/CSE151B_Kaggle

from google.colab import files
files.download("/content/CSE151B_Kaggle.zip")


### Only if in DataHub

In [ ]:
# Install uv
!wget -qO- https://astral.sh/uv/install.sh | sh

!export PATH="/home/ugheewala/.local/bin:$PATH"

# Create a virtual environment
!/home/ugheewala/.local/bin/uv venv .venv --seed --clear

# Install dependencies — this is fast thanks to uv's parallel resolver
!.venv/bin/python -m pip install prettyprint sympy numpy pandas matplotlib transformers accelerate vllm tqdm bitsandbytes antlr4-python3-runtime==4.11.1 ipykernel jupyter

# Install Jupyter Kernel
!.venv/bin/python -m ipykernel install --user --name cse151b --display-name "Python (cse151b)"

print("Done. Restart the kernel before proceeding.")
print("Selection process: on top right, click on current kernel '(ususally named python)' -> 'select another kernel' -> 'Jupyter Kernel' -> 'Python (cse151b)'.")

In [ ]:
# !/home/<name>/.local/bin/uv pip install --python .venv/bin/python \
#     "numpy<2" \
#     "torch==2.1.2+cu118" \
#     "transformers==4.51.3" \
#     "accelerate==0.34.2" \
#     "huggingface_hub>=0.23.0" \
#     "safetensors" \
#     "sentencepiece" \
#     "tqdm" \
#     "pandas" \
#     "matplotlib" \
#     "sympy" \
#     "antlr4-python3-runtime==4.11.1" \
#     --extra-index-url https://download.pytorch.org/whl/cu118

In [ ]:
# !/home/<name>/.local/bin/uv pip install --python .venv/bin/python \
#     "nvidia-cusparse-cu11" \
#     "nvidia-cublas-cu11" \
#     "nvidia-cuda-runtime-cu11" \
#     "nvidia-cudnn-cu11"

In [ ]:
# activate venv after installation. This needs to be run everytime.
!source ./.venv/bin/activate

In [ ]:
!/home/<name>/.local/bin/uv pip install --python .venv/bin/python \
    "numpy<2" \
    "torch==2.3.1+cu121" \
    "torchvision==0.18.1+cu121" \
    "torchaudio==2.3.1+cu121" \
    --extra-index-url https://download.pytorch.org/whl/cu121

In [ ]:
!/home/<name>/.local/bin/uv pip install --python .venv/bin/python \
    "transformers==4.51.3" \
    "accelerate>=0.30.0" \
    "huggingface_hub>=0.23.0" \
    "safetensors" \
    "sentencepiece" \
    "tokenizers==0.21.4" \
    "sympy" \
    "pandas" \
    "matplotlib" \
    "tqdm" \
    "prettyprint" \
    "antlr4-python3-runtime==4.11.1" \
    "ipykernel" \
    "jupyter"

In [ ]:
!/home/<name>/.local/bin/uv pip install --python .venv/bin/python \
    "bitsandbytes==0.45.5"

## 2. Imports & Configuration

In [3]:
import os
import sys
import json
import time
import csv
import subprocess
from pathlib import Path
from pprint import pprint

import pandas as pd
import numpy as np
from tqdm.auto import tqdm

# ── Configuration ─────────────────────────────────────────────────────────────
MODEL_ID    = "Qwen/Qwen3-4B-Thinking-2507"
GPU_ID      = "0"                    # CUDA_VISIBLE_DEVICES
PUBLIC_DATA_PATH   = "data/public.jsonl"
PRIVATE_DATA_PATH   = "data/private.jsonl"
OUTPUT_PATH = "results/starter_results.jsonl"

PROJECT_ROOT = Path.cwd()

RESULTS_DIR = PROJECT_ROOT / "results"
BASELINE1_DIR = RESULTS_DIR / "baseline1_weakest"
BASELINE1_DIR.mkdir(parents=True, exist_ok=True)

VAL_FRAC = 0.20
SPLIT_SEED = 414

CACHE_DIR = None
HF_HOME_DIR = None

MAX_INPUT_TOKENS = 8192 #4096
MAX_MODEL_LEN = 8192 #4096

MAX_NEW_TOKENS_SMOKE = 8192 #32768
MAX_NEW_TOKENS_BASELINE = 8192 #1024

INFERENCE_BACKEND = "vllm"

BATCH_SIZE = 32
LOAD_IN_4BIT = False

MAX_TOKENS = 8192

os.environ["CUDA_VISIBLE_DEVICES"] = GPU_ID

if HF_HOME_DIR is not None:
    os.environ["HF_HOME"] = str(HF_HOME_DIR)

if CACHE_DIR is not None:
    Path(CACHE_DIR).mkdir(parents=True, exist_ok=True)

print("HF_HOME      :", os.environ.get("HF_HOME"))
print("HF_HUB_CACHE :", os.environ.get("HF_HUB_CACHE"))
print("cache_dir    :", CACHE_DIR)

HF_HOME      : None
HF_HUB_CACHE : None
cache_dir    : None


In [5]:
import torch

print(f"CUDA_VISIBLE_DEVICES (Env): {os.environ.get('CUDA_VISIBLE_DEVICES')}")

cuda_available = torch.cuda.is_available()
print(f"Is CUDA available? {cuda_available}")

if cuda_available:
    print(f"Current Device: {torch.cuda.current_device()}")
    print(f"Device Name: {torch.cuda.get_device_name(0)}")
    device = torch.device("cuda")
else:
    print("PyTorch still can't see the GPU.")
    device = torch.device("cpu")

CUDA_VISIBLE_DEVICES (Env): 0
Is CUDA available? True
Current Device: 0
Device Name: NVIDIA A100-SXM4-80GB


In [6]:
# import site

# roots = [Path(p) for p in site.getsitepackages()]
# matches = []

# for root in roots:
#     if root.exists():
#         matches.extend(root.rglob("libcusparse.so*"))

# for m in matches:
#     print(m)

In [7]:
# wanted_libs = {
#     "libcusparse.so",
#     "libcublas.so",
#     "libcudart.so",
#     "libcudnn.so",
# }

# lib_dirs = []

# for root in map(Path, site.getsitepackages()):
#     if not root.exists():
#         continue

#     for lib in wanted_libs:
#         for match in root.rglob(lib + "*"):
#             lib_dir = str(match.parent)
#             if lib_dir not in lib_dirs:
#                 lib_dirs.append(lib_dir)

# VENV = Path("/home/ugheewala/private/CSE151B_Kaggle/.venv")
# SITE = VENV / "lib/python3.11/site-packages"

# cuda11_dirs = [
#     SITE / "nvidia/cusparse/lib",
#     SITE / "nvidia/cublas/lib",
#     SITE / "nvidia/cuda_runtime/lib",
#     SITE / "nvidia/cudnn/lib",
#     SITE / "torch/lib",
# ]

# cuda11_dirs = [str(p) for p in cuda11_dirs if p.exists()]

# path_line = ":".join(cuda11_dirs)

# print("Add this before starting the notebook/kernel:")
# print(f'export LD_LIBRARY_PATH="{path_line}:$LD_LIBRARY_PATH"')

In [8]:
# import os
# import subprocess
# from pathlib import Path

# VENV = Path("/home/ugheewala/private/CSE151B_Kaggle/.venv")
# SITE = VENV / "lib/python3.11/site-packages"

# cuda11_dirs = [
#     SITE / "nvidia/cusparse/lib",
#     SITE / "nvidia/cublas/lib",
#     SITE / "nvidia/cuda_runtime/lib",
#     SITE / "nvidia/cudnn/lib",
#     SITE / "torch/lib",
# ]

# env = os.environ.copy()
# env["LD_LIBRARY_PATH"] = ":".join(str(p) for p in cuda11_dirs if p.exists()) + ":" + env.get("LD_LIBRARY_PATH", "")

# subprocess.run(
#     [str(VENV / "bin/python"), "-m", "bitsandbytes"],
#     env=env,
# )

In [9]:
# import json
# from pathlib import Path

# kernel_json = Path("/home/ugheewala/.local/share/jupyter/kernels/cse151b/kernel.json")

# with open(kernel_json, "r") as f:
#     spec = json.load(f)

# ld_library_path = (
#     "/home/ugheewala/private/CSE151B_Kaggle/.venv/lib/python3.11/site-packages/nvidia/cusparse/lib:"
#     "/home/ugheewala/private/CSE151B_Kaggle/.venv/lib/python3.11/site-packages/nvidia/cublas/lib:"
#     "/home/ugheewala/private/CSE151B_Kaggle/.venv/lib/python3.11/site-packages/nvidia/cuda_runtime/lib:"
#     "/home/ugheewala/private/CSE151B_Kaggle/.venv/lib/python3.11/site-packages/nvidia/cudnn/lib:"
#     "/home/ugheewala/private/CSE151B_Kaggle/.venv/lib/python3.11/site-packages/torch/lib:"
#     "${LD_LIBRARY_PATH}"
# )

# spec.setdefault("env", {})
# spec["env"]["LD_LIBRARY_PATH"] = ld_library_path
# spec["env"]["BNB_CUDA_VERSION"] = "118"

# with open(kernel_json, "w") as f:
#     json.dump(spec, f, indent=2)

# print(kernel_json)
# print(json.dumps(spec, indent=2))

In [10]:
import transformers

print("torch:", torch.__version__)
print("cuda:", torch.version.cuda)
print("cuda available:", torch.cuda.is_available())
print("transformers:", transformers.__version__)

torch: 2.10.0+cu128
cuda: 12.8
cuda available: True
transformers: 4.57.6


In [11]:
try:
    import bitsandbytes as bnb
    print("bitsandbytes:", bnb.__version__)
except Exception as e:
    print("bitsandbytes import failed:", repr(e))

bitsandbytes: 0.49.2


In [12]:
from transformers.utils import is_torch_available, is_bitsandbytes_available

print("is_torch_available:", is_torch_available())
print("is_bitsandbytes_available:", is_bitsandbytes_available())

is_torch_available: True
is_bitsandbytes_available: True


In [13]:
try:
    import vllm
    print("vLLM version:", vllm.__version__)
    from vllm import LLM, SamplingParams
    print("vLLM import OK")
except Exception as e:
    print("vLLM import FAILED:")
    raise

vLLM version: 0.18.0
vLLM import OK


In [4]:
from transformers import AutoTokenizer
from tqdm import tqdm

from baseline.datasets import load_public_splits, load_private_set
from baseline.generation import GenerationConfig
from baseline.prompt_sets import build_prompt_texts
from baseline.modeling import ModelConfig, load_transformers_model, predownload_model, load_model, detect_gpu_info
from baseline.scoring import load_judger, score_one, summarize_results
from baseline.progress_viz import RunProgressDashboard
from prompting.prompt_chain import build_prompt_chain
from baseline.runner import run_problem_set, benchmark_batch_sizes, write_report

In [7]:
gpu_info = detect_gpu_info()
pprint(gpu_info.to_dict())

{'capability': (8, 0),
 'cuda_available': True,
 'device_count': 1,
 'device_name': 'NVIDIA A100-SXM4-80GB'}


## 3. Load the Dataset

The dataset is stored as newline-delimited JSON (`.jsonl`). Each line is one question with the following fields:

| Field | Description |
|---|---|
| `id` | Unique question identifier |
| `question` | Problem statement |
| `options` | List of answer choices - present for **MCQ**, absent for **free-form** |
| `answer` | Ground-truth answer (letter for MCQ, value/list for free-form) |

In [5]:
splits = load_public_splits(PUBLIC_DATA_PATH, val_frac=VAL_FRAC, seed=SPLIT_SEED)

train_set = splits["train"]
val_set = splits["val"]
public_set = splits["public"]
private_set = load_private_set(PRIVATE_DATA_PATH)

print("Train summary:")
pprint(train_set.summary())

print("\nValidation summary:")
pprint(val_set.summary())

print("\nPublic summary:")
pprint(public_set.summary())

print("\nPrivate summary:")
pprint(private_set.summary())

Train summary:
{'n': 901,
 'n_answered': 901,
 'n_free_form': 601,
 'n_mcq': 300,
 'name': 'public_train'}

Validation summary:
{'n': 225,
 'n_answered': 225,
 'n_free_form': 150,
 'n_mcq': 75,
 'name': 'public_val'}

Public summary:
{'n': 1126,
 'n_answered': 1126,
 'n_free_form': 751,
 'n_mcq': 375,
 'name': 'public'}

Private summary:
{'n': 943, 'n_answered': 0, 'n_free_form': 643, 'n_mcq': 300, 'name': 'private'}


In [6]:
prompt_chain = build_prompt_chain(strategy_name="baseline")

for label, problem_set in [("train", train_set), ("val", val_set), ("private", private_set)]:
    problem = problem_set.problems()[0]
    spec = prompt_chain.build_spec(problem)

    print("=" * 80)
    print(label, "id=", problem.id, "template=", spec.name)
    print("metadata:", spec.metadata)
    print("generation_hints:", spec.generation_hints)
    print(spec.to_messages()[0]["content"][:300])
    print("--- user ---")
    print(spec.to_messages()[-1]["content"][:500])

train id= 499 template= baseline_mcq
metadata: {'answer_format': 'mcq', 'category': 'general_math', 'qwen_categories': ['general_math'], 'derived_rules': [], 'rule_annotations': {}, 'strategy_label': 'baseline_weakest', 'route_template_name': 'mcq', 'strategy_name': 'baseline', 'route_name': 'mcq', 'tags': ['general_math']}
generation_hints: {'temperature': 0.6, 'top_p': 0.95}
You are an expert mathematician. Read the problem and the answer choices below, then select the single best answer. Output only the letter of your chosen option inside \boxed{}, e.g. \boxed{C}.
--- user ---
We now define an algorithm: The definition of a(n) is the least odd number k such that k * 2^n + 1 is a prime number. Given the input x_list (a series of values): [70, 71, 72, 73, 74, 75, 76, 77, 78, 79], determine the corresponding output sequence y_list.

Answer choices:
A. [44, 43, 129, 26, 63, 1, 90, 33, 22, 243]
B. [37, 35, 122, 19, 64, 10, 96, 26, 20, 245]
C. [38, 40, 128, 22, 71, 3, 91, 28, 14, 248]
D. 

## 4. Modeling

In [7]:
model_config = ModelConfig(
    model_id=MODEL_ID,
    backend=INFERENCE_BACKEND,
    cache_dir=CACHE_DIR,
    gpu_id=GPU_ID,
    max_input_tokens=MAX_INPUT_TOKENS,
    max_model_len=MAX_MODEL_LEN,
    dtype="bfloat16",
    torch_dtype="bfloat16",
    load_in_4bit=True,
    device_map="auto",
    low_cpu_mem_usage=True,
    gpu_memory_utilization=0.85,
    max_num_seqs=256,
    max_num_batched_tokens=MAX_TOKENS,
    reuse_loaded=True,
)

t0 = time.perf_counter()
model_bundle = load_model(model_config)
print(f"Model load/reuse time: {time.perf_counter() - t0:.2f} sec")
print("Backend:", model_bundle.backend)
print("Device:", model_bundle.device())

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


INFO 05-31 00:15:57 [utils.py:233] non-default args: {'trust_remote_code': True, 'load_format': 'bitsandbytes', 'dtype': 'bfloat16', 'max_model_len': 8192, 'enable_prefix_caching': False, 'gpu_memory_utilization': 0.85, 'max_num_batched_tokens': 8192, 'max_num_seqs': 256, 'disable_log_stats': True, 'quantization': 'bitsandbytes', 'model': 'Qwen/Qwen3-4B-Thinking-2507'}
INFO 05-31 00:15:58 [model.py:533] Resolved architecture: Qwen3ForCausalLM
INFO 05-31 00:15:58 [model.py:1582] Using max model len 8192
INFO 05-31 00:15:58 [scheduler.py:231] Chunked prefill is enabled with max_num_batched_tokens=8192.
INFO 05-31 00:15:59 [vllm.py:754] Asynchronous scheduling is enabled.
WARNING 05-31 00:16:00 [system_utils.py:152] We must use the `spawn` multiprocessing start method. Overriding VLLM_WORKER_MULTIPROC_METHOD to 'spawn'. See https://docs.vllm.ai/en/latest/usage/troubleshooting.html#python-multiprocessing for more information. Reasons: CUDA is initialized
INFO 05-31 00:17:14 [llm.py:391] Su

# Baseline 1: Naive

In [ ]:
baseline1_prompt_chain = build_prompt_chain(strategy_name="baseline")

smoke_generation_config = GenerationConfig(
    max_new_tokens=MAX_NEW_TOKENS_SMOKE,
    temperature=0.6,
    top_p=0.95,
    top_k=20,
    min_p=0.0,
    repetition_penalty=1.0,
    presence_penalty=0.0,
    do_sample=True,
)

baseline1_generation_config = GenerationConfig(
    max_new_tokens=MAX_NEW_TOKENS_BASELINE,
    temperature=0.6,
    top_p=0.95,
    top_k=20,
    min_p=0.0,
    repetition_penalty=1.0,
    presence_penalty=0.0,
    do_sample=True,
)

In [ ]:
batch_report = benchmark_batch_sizes(
    problem_set=train_set,
    model_bundle=model_bundle,
    batch_sizes=[1, 2, 4, 8, 16, 32, 64],
    prompt_chain=baseline1_prompt_chain,
    generation_config=smoke_generation_config,
    sample_size=16,
    score=False,
    show_progress=False,
)

pd.DataFrame(batch_report["rows"])

In [ ]:
# Train split smoke run

baseline1_train_result = run_problem_set(
    problem_set=train_set,
    model_bundle=model_bundle,
    prompt_chain=baseline1_prompt_chain,
    generation_config=baseline1_generation_config,
    batch_size=BATCH_SIZE,
    limit=32,
    score=True,
    output_jsonl_path=BASELINE1_DIR / "train_results.jsonl",
    report_json_path=BASELINE1_DIR / "train_report.json",
)

pprint(baseline1_train_result.report)

In [ ]:
"""
'generation_config': {'do_sample': True,
                       'max_new_tokens': 1024,
                       'min_p': 0.0,
                       'presence_penalty': 0.0,
                       'repetition_penalty': 1.0,
                       'temperature': 0.6,
                       'top_k': 20,
                       'top_p': 0.95},
 'output_jsonl_path': '/content/CSE151B_Kaggle/results/baseline1_weakest/train_results.jsonl',
 'problem_set': {'n': 32,
                 'n_answered': 32,
                 'n_free_form': 13,
                 'n_mcq': 19,
                 'name': 'public_train_head32'},
 'score_available': True,
 'submission_csv_path': None,
 'summary': {'free_form_acc': 0.3076923076923077,
             'mcq_acc': 0.05263157894736842,
             'n_correct': 5,
             'n_free_form': 13,
             'n_mcq': 19,
             'n_scored': 32,
             'overall_acc': 0.15625},
 'timings': {'generation_sec': 11.5697886199996,
             'prompt_build_sec': 0.002926810000644764,
             'scoring_sec': 0.7455679300001066}}
"""

In [ ]:
"""
'generation_config': {'do_sample': True,
                       'max_new_tokens': 32768,
                       'min_p': 0.0,
                       'presence_penalty': 0.0,
                       'repetition_penalty': 1.0,
                       'temperature': 0.6,
                       'top_k': 20,
                       'top_p': 0.95},
 'output_jsonl_path': '/content/CSE151B_Kaggle/CSE151B_Kaggle/results/baseline1_weakest/train_results.jsonl',
 'problem_set': {'n': 32,
                 'n_answered': 32,
                 'n_free_form': 13,
                 'n_mcq': 19,
                 'name': 'public_train_head32'},
 'score_available': True,
 'submission_csv_path': None,
 'summary': {'free_form_acc': 0.6923076923076923,
             'mcq_acc': 0.7368421052631579,
             'n_correct': 23,
             'n_free_form': 13,
             'n_mcq': 19,
             'n_scored': 32,
             'overall_acc': 0.71875},
 'timings': {'generation_sec': 461.65091967599983,
             'prompt_build_sec': 0.0029734310001003905,
             'scoring_sec': 1.0903050570000232}}
"""

In [ ]:
# Validation run

baseline1_val_result = run_problem_set(
    problem_set=val_set,
    model_bundle=model_bundle,
    prompt_chain=baseline1_prompt_chain,
    generation_config=baseline1_generation_config,
    batch_size=BATCH_SIZE,
    limit=None,
    score=True,
    output_jsonl_path=BASELINE1_DIR / "val_results.jsonl",
    report_json_path=BASELINE1_DIR / "val_report.json",
)

pprint(baseline1_val_result.report)

In [ ]:
"""
 'generation_config': {'do_sample': True,
                       'max_new_tokens': 1024,
                       'min_p': 0.0,
                       'presence_penalty': 0.0,
                       'repetition_penalty': 1.0,
                       'temperature': 0.6,
                       'top_k': 20,
                       'top_p': 0.95},
 'output_jsonl_path': '/content/CSE151B_Kaggle/results/baseline1_weakest/val_results.jsonl',
 'problem_set': {'n': 225,
                 'n_answered': 225,
                 'n_free_form': 150,
                 'n_mcq': 75,
                 'name': 'public_val'},
 'score_available': True,
 'submission_csv_path': None,
 'summary': {'free_form_acc': 0.12666666666666668,
             'mcq_acc': 0.08,
             'n_correct': 25,
             'n_free_form': 150,
             'n_mcq': 75,
             'n_scored': 225,
             'overall_acc': 0.1111111111111111},
 'timings': {'generation_sec': 87.61873525499959,
             'prompt_build_sec': 0.01957373799996276,
             'scoring_sec': 7.864173332000064}}
"""

In [ ]:
"""
'generation_config': {'do_sample': True,
                       'max_new_tokens': 32768,
                       'min_p': 0.0,
                       'presence_penalty': 0.0,
                       'repetition_penalty': 1.0,
                       'temperature': 0.6,
                       'top_k': 20,
                       'top_p': 0.95},
 'output_jsonl_path': '/content/CSE151B_Kaggle/CSE151B_Kaggle/results/baseline1_weakest/val_results.jsonl',
 'problem_set': {'n': 225,
                 'n_answered': 225,
                 'n_free_form': 150,
                 'n_mcq': 75,
                 'name': 'public_val'},
 'score_available': True,
 'submission_csv_path': None,
 'summary': {'free_form_acc': 0.54,
             'mcq_acc': 0.8266666666666667,
             'n_correct': 143,
             'n_free_form': 150,
             'n_mcq': 75,
             'n_scored': 225,
             'overall_acc': 0.6355555555555555},
 'timings': {'generation_sec': 2392.0188707439997,
             'prompt_build_sec': 0.038322351000260824,
             'scoring_sec': 16.22751727099967}}
"""

In [ ]:
# Test run

PRIVATE_LIMIT = None

baseline1_private_result = run_problem_set(
    problem_set=private_set,
    model_bundle=model_bundle,
    prompt_chain=baseline1_prompt_chain,
    generation_config=baseline1_generation_config,
    batch_size=BATCH_SIZE,
    limit=None,
    score=False,
    output_jsonl_path=BASELINE1_DIR / "private_results.jsonl",
    submission_csv_path=BASELINE1_DIR / "submission.csv",
    report_json_path=BASELINE1_DIR / "private_report.json",
)

pprint(baseline1_private_result.report)

In [ ]:
"""
'generation_config': {'do_sample': True,
                       'max_new_tokens': 1024,
                       'min_p': 0.0,
                       'presence_penalty': 0.0,
                       'repetition_penalty': 1.0,
                       'temperature': 0.6,
                       'top_k': 20,
                       'top_p': 0.95},
 'output_jsonl_path': '/content/CSE151B_Kaggle/results/baseline1_weakest/private_results.jsonl',
 'problem_set': {'n': 943,
                 'n_answered': 0,
                 'n_free_form': 643,
                 'n_mcq': 300,
                 'name': 'private'},
 'score_available': False,
 'submission_csv_path': '/content/CSE151B_Kaggle/results/baseline1_weakest/submission.csv',
 'summary': {'free_form_acc': None,
             'mcq_acc': None,
             'n_correct': 0,
             'n_free_form': 0,
             'n_mcq': 0,
             'n_scored': 0,
             'overall_acc': None},
 'timings': {'generation_sec': 339.3967333410001,
             'prompt_build_sec': 0.07438255999932153,
             'scoring_sec': 0.0008840780001264648}}
"""

In [ ]:
"""
'generation_config': {'do_sample': True,
                       'max_new_tokens': 32768,
                       'min_p': 0.0,
                       'presence_penalty': 0.0,
                       'repetition_penalty': 1.0,
                       'temperature': 0.6,
                       'top_k': 20,
                       'top_p': 0.95},
 'output_jsonl_path': '/content/CSE151B_Kaggle/CSE151B_Kaggle/results/baseline1_weakest/private_results.jsonl',
 'problem_set': {'n': 943,
                 'n_answered': 0,
                 'n_free_form': 643,
                 'n_mcq': 300,
                 'name': 'private'},
 'score_available': False,
 'submission_csv_path': '/content/CSE151B_Kaggle/CSE151B_Kaggle/results/baseline1_weakest/submission.csv',
 'summary': {'free_form_acc': None,
             'mcq_acc': None,
             'n_correct': 0,
             'n_free_form': 0,
             'n_mcq': 0,
             'n_scored': 0,
             'overall_acc': None},
 'timings': {'generation_sec': 12278.958291005998,
             'prompt_build_sec': 0.07650407100027223,
             'scoring_sec': 0.002097485998092452}}
"""

In [ ]:
# At the end to download shit

!zip -r B1_results.zip /content/CSE151B_Kaggle/CSE151B_Kaggle/results/baseline1_weakest

from google.colab import files
files.download('B1_results.zip')


In [ ]:
submission_path = BASELINE1_DIR / "submission.csv"

if submission_path.exists():
    sub_df = pd.read_csv(submission_path)
    print(sub_df.shape)
    display(sub_df.head())
    print("Columns:", list(sub_df.columns))
else:
    print("No submission file yet. Run the private test cell first.")

# Baseline 2: Output hardening

In [ ]:
from baseline.baseline2_runner import run_baseline2_problem_set
from baseline.generation import GenerationConfig

BASELINE2_DIR = RESULTS_DIR / "baseline2_prompt_format"
BASELINE2_DIR.mkdir(parents=True, exist_ok=True)

baseline2_generation_config = GenerationConfig(
    max_new_tokens=MAX_TOKENS,#4096,
    temperature=0.6,
    top_p=0.95,
    top_k=20,
    min_p=0.0,
    repetition_penalty=1.0,
    presence_penalty=0.0,
    do_sample=True,
)

baseline2_retry_generation_config = GenerationConfig(
    max_new_tokens=MAX_TOKENS,#1024,
    temperature=0.0,
    top_p=1.0,
    top_k=-1,
    min_p=0.0,
    repetition_penalty=1.0,
    presence_penalty=0.0,
    do_sample=False,
)

In [ ]:
print("Model backend:", model_bundle.backend)
print("Model max_model_len:", model_bundle.config.max_model_len)
print("Generation max_new_tokens:", baseline2_generation_config.max_new_tokens)
print("Retry max_new_tokens:", baseline2_retry_generation_config.max_new_tokens)

from baseline.prompt_sets import build_prompt_texts
from prompting.prompt_chain import build_prompt_chain

prompt_chain = build_prompt_chain(strategy_name="baseline2")
prompt_rows = build_prompt_texts(val_set.head(5), model_bundle.tokenizer, prompt_chain=prompt_chain)

for row in prompt_rows:
    toks = model_bundle.tokenizer.encode(row["prompt_text"])
    print(row["id"], "prompt_tokens:", len(toks))

for row in prompt_rows:
  prompt_tokens = len(model_bundle.tokenizer.encode(row["prompt_text"]))
  available = model_bundle.config.max_model_len - prompt_tokens
  effective_new_token_cap = min(baseline2_generation_config.max_new_tokens, available)

  print({
      "id": row["id"],
      "prompt_tokens": prompt_tokens,
      "max_model_len": model_bundle.config.max_model_len,
      "configured_max_new_tokens": baseline2_generation_config.max_new_tokens,
      "effective_new_token_cap": effective_new_token_cap,
  })

In [ ]:
baseline2_train_result = run_baseline2_problem_set(
    problem_set=train_set,
    model_bundle=model_bundle,
    generation_config=baseline2_generation_config,
    retry_generation_config=baseline2_retry_generation_config,
    batch_size=BATCH_SIZE,
    limit=32,
    score=True,
    output_jsonl_path=BASELINE2_DIR / "train_results.jsonl",
    debug_jsonl_path=BASELINE2_DIR / "train_debug.jsonl",
    report_json_path=BASELINE2_DIR / "train_report.json",
    show_progress=True,
)

pprint(baseline2_train_result.report)

In [ ]:
"""
'generation_config': {'do_sample': True,
                       'max_new_tokens': 1024,
                       'min_p': 0.0,
                       'presence_penalty': 0.0,
                       'repetition_penalty': 1.0,
                       'temperature': 0.6,
                       'top_k': 20,
                       'top_p': 0.95},
 'output_jsonl_path': '/content/CSE151B_Kaggle/CSE151B_Kaggle/results/baseline2_prompt_format/train_results.jsonl',
 'problem_set': {'n': 32,
                 'n_answered': 32,
                 'n_free_form': 13,
                 'n_mcq': 19,
                 'name': 'public_train_head32'},
 'schema_accuracy': {'retry_used_accuracy': 0.6,
                     'retry_used_n': 5,
                     'schema_invalid_accuracy': 0.08333333333333333,
                     'schema_invalid_n': 24,
                     'schema_valid_accuracy': 0.875,
                     'schema_valid_n': 8},
 'score_available': True,
 'submission_csv_path': None,
 'summary': {'free_form_acc': 0.5384615384615384,
             'mcq_acc': 0.10526315789473684,
             'n_correct': 9,
             'n_free_form': 13,
             'n_mcq': 19,
             'n_scored': 32,
             'overall_acc': 0.28125},
 'timings': {'generation_sec': 11.575837816000103,
             'prompt_build_sec': 0.023429658999930325,
             'retry_generation_sec': 11.110912010999982,
             'scoring_sec': 0.8481839019998461}}
"""

In [ ]:
"""
'generation_config': {'do_sample': True,
                       'max_new_tokens': 4096,
                       'min_p': 0.0,
                       'presence_penalty': 0.0,
                       'repetition_penalty': 1.0,
                       'temperature': 0.6,
                       'top_k': 20,
                       'top_p': 0.95},
 'output_jsonl_path': '/content/CSE151B_Kaggle/CSE151B_Kaggle/results/baseline2_prompt_format/train_results.jsonl',
 'problem_set': {'n': 32,
                 'n_answered': 32,
                 'n_free_form': 13,
                 'n_mcq': 19,
                 'name': 'public_train_head32'},
 'retry_generation_config': {'do_sample': False,
                             'max_new_tokens': 1024,
                             'min_p': 0.0,
                             'presence_penalty': 0.0,
                             'repetition_penalty': 1.0,
                             'temperature': 0.0,
                             'top_k': -1,
                             'top_p': 1.0},
 'schema_accuracy': {'retry_used_accuracy': 0.0,
                     'retry_used_n': 1,
                     'sanitized_accuracy': 0.6,
                     'sanitized_n': 5,
                     'schema_invalid_accuracy': 0.0,
                     'schema_invalid_n': 7,
                     'schema_valid_accuracy': 0.56,
                     'schema_valid_n': 25},
 'score_available': True,
 'submission_csv_path': None,
 'summary': {'free_form_acc': 0.46153846153846156,
             'mcq_acc': 0.42105263157894735,
             'n_correct': 14,
             'n_free_form': 13,
             'n_mcq': 19,
             'n_scored': 32,
             'overall_acc': 0.4375},
 'timings': {'generation_sec': 46.100986020000164,
             'prompt_build_sec': 0.0028652999999394524,
             'retry_generation_sec': 8.265795843999967,
             'scoring_sec': 1.19583538400002}}
"""

In [ ]:
"""
'generation_config': {'do_sample': True,
                       'max_new_tokens': 32768,
                       'min_p': 0.0,
                       'presence_penalty': 0.0,
                       'repetition_penalty': 1.0,
                       'temperature': 0.6,
                       'top_k': 20,
                       'top_p': 0.95},
 'output_jsonl_path': '/content/CSE151B_Kaggle/CSE151B_Kaggle/results/baseline2_prompt_format/train_results.jsonl',
 'problem_set': {'n': 32,
                 'n_answered': 32,
                 'n_free_form': 13,
                 'n_mcq': 19,
                 'name': 'public_train_head32'},
 'retry_generation_config': {'do_sample': False,
                             'max_new_tokens': 32768,
                             'min_p': 0.0,
                             'presence_penalty': 0.0,
                             'repetition_penalty': 1.0,
                             'temperature': 0.0,
                             'top_k': -1,
                             'top_p': 1.0},
 'schema_accuracy': {'retry_used_accuracy': None,
                     'retry_used_n': 0,
                     'sanitized_accuracy': None,
                     'sanitized_n': 0,
                     'schema_invalid_accuracy': 0.0,
                     'schema_invalid_n': 2,
                     'schema_valid_accuracy': 0.7,
                     'schema_valid_n': 30},
 'score_available': True,
 'submission_csv_path': None,
 'summary': {'free_form_acc': 0.6923076923076923,
             'mcq_acc': 0.631578947368421,
             'n_correct': 21,
             'n_free_form': 13,
             'n_mcq': 19,
             'n_scored': 32,
             'overall_acc': 0.65625},
 'timings': {'generation_sec': 165.0723143110008,
             'prompt_build_sec': 0.0030510440010402817,
             'retry_generation_sec': 0.5391863780023414,
             'scoring_sec': 0.7430123269987234}}
"""

In [ ]:
baseline2_val_result = run_baseline2_problem_set(
    problem_set=val_set,
    model_bundle=model_bundle,
    generation_config=baseline2_generation_config,
    retry_generation_config=baseline2_retry_generation_config,
    batch_size=BATCH_SIZE,
    limit=None,
    score=True,
    output_jsonl_path=BASELINE2_DIR / "val_results.jsonl",
    debug_jsonl_path=BASELINE2_DIR / "val_debug.jsonl",
    report_json_path=BASELINE2_DIR / "val_report.json",
    show_progress=True,
)

pprint(baseline2_val_result.report)

In [ ]:
"""
'generation_config': {'do_sample': True,
                       'max_new_tokens': 1024,
                       'min_p': 0.0,
                       'presence_penalty': 0.0,
                       'repetition_penalty': 1.0,
                       'temperature': 0.6,
                       'top_k': 20,
                       'top_p': 0.95},
 'output_jsonl_path': '/content/CSE151B_Kaggle/CSE151B_Kaggle/results/baseline2_prompt_format/val_results.jsonl',
 'problem_set': {'n': 225,
                 'n_answered': 225,
                 'n_free_form': 150,
                 'n_mcq': 75,
                 'name': 'public_val'},
 'schema_accuracy': {'retry_used_accuracy': 0.6590909090909091,
                     'retry_used_n': 44,
                     'schema_invalid_accuracy': 0.17333333333333334,
                     'schema_invalid_n': 150,
                     'schema_valid_accuracy': 0.76,
                     'schema_valid_n': 75},
 'score_available': True,
 'submission_csv_path': None,
 'summary': {'free_form_acc': 0.4666666666666667,
             'mcq_acc': 0.17333333333333334,
             'n_correct': 83,
             'n_free_form': 150,
             'n_mcq': 75,
             'n_scored': 225,
             'overall_acc': 0.3688888888888889},
 'timings': {'generation_sec': 82.84365255199987,
             'prompt_build_sec': 0.02054123199991409,
             'retry_generation_sec': 69.44308633399987,
             'scoring_sec': 14.325112181999884}}
"""

In [ ]:
"""
'generation_config': {'do_sample': True,
                       'max_new_tokens': 4096,
                       'min_p': 0.0,
                       'presence_penalty': 0.0,
                       'repetition_penalty': 1.0,
                       'temperature': 0.6,
                       'top_k': 20,
                       'top_p': 0.95},
 'output_jsonl_path': '/content/CSE151B_Kaggle/CSE151B_Kaggle/results/baseline2_prompt_format/val_results.jsonl',
 'problem_set': {'n': 225,
                 'n_answered': 225,
                 'n_free_form': 150,
                 'n_mcq': 75,
                 'name': 'public_val'},
 'retry_generation_config': {'do_sample': False,
                             'max_new_tokens': 1024,
                             'min_p': 0.0,
                             'presence_penalty': 0.0,
                             'repetition_penalty': 1.0,
                             'temperature': 0.0,
                             'top_k': -1,
                             'top_p': 1.0},
 'schema_accuracy': {'retry_used_accuracy': 0.25,
                     'retry_used_n': 8,
                     'sanitized_accuracy': 0.3142857142857143,
                     'sanitized_n': 35,
                     'schema_invalid_accuracy': 0.05405405405405406,
                     'schema_invalid_n': 37,
                     'schema_valid_accuracy': 0.6063829787234043,
                     'schema_valid_n': 188},
 'score_available': True,
 'submission_csv_path': None,
 'summary': {'free_form_acc': 0.56,
             'mcq_acc': 0.4266666666666667,
             'n_correct': 116,
             'n_free_form': 150,
             'n_mcq': 75,
             'n_scored': 225,
             'overall_acc': 0.5155555555555555},
 'timings': {'generation_sec': 341.6052394200001,
             'prompt_build_sec': 0.023227069000085976,
             'retry_generation_sec': 21.939288180000176,
             'scoring_sec': 19.803487271999984}}
"""

In [ ]:
"""
'generation_config': {'do_sample': True,
                       'max_new_tokens': 32768,
                       'min_p': 0.0,
                       'presence_penalty': 0.0,
                       'repetition_penalty': 1.0,
                       'temperature': 0.6,
                       'top_k': 20,
                       'top_p': 0.95},
 'output_jsonl_path': '/content/CSE151B_Kaggle/CSE151B_Kaggle/results/baseline2_prompt_format/val_results.jsonl',
 'problem_set': {'n': 225,
                 'n_answered': 225,
                 'n_free_form': 150,
                 'n_mcq': 75,
                 'name': 'public_val'},
 'retry_generation_config': {'do_sample': False,
                             'max_new_tokens': 32768,
                             'min_p': 0.0,
                             'presence_penalty': 0.0,
                             'repetition_penalty': 1.0,
                             'temperature': 0.0,
                             'top_k': -1,
                             'top_p': 1.0},
 'schema_accuracy': {'retry_used_accuracy': 1.0,
                     'retry_used_n': 2,
                     'sanitized_accuracy': 1.0,
                     'sanitized_n': 3,
                     'schema_invalid_accuracy': 0.07692307692307693,
                     'schema_invalid_n': 13,
                     'schema_valid_accuracy': 0.660377358490566,
                     'schema_valid_n': 212},
 'score_available': True,
 'submission_csv_path': None,
 'summary': {'free_form_acc': 0.56,
             'mcq_acc': 0.76,
             'n_correct': 141,
             'n_free_form': 150,
             'n_mcq': 75,
             'n_scored': 225,
             'overall_acc': 0.6266666666666667},
 'timings': {'generation_sec': 1119.7211674520004,
             'prompt_build_sec': 0.020980745000997558,
             'retry_generation_sec': 20.915390042999206,
             'scoring_sec': 17.43489298500208}}
"""

In [ ]:
baseline2_private_result = run_baseline2_problem_set(
    problem_set=private_set,
    model_bundle=model_bundle,
    generation_config=baseline2_generation_config,
    retry_generation_config=baseline2_retry_generation_config,
    batch_size=BATCH_SIZE,
    limit=None,
    score=False,
    output_jsonl_path=BASELINE2_DIR / "private_results.jsonl",
    debug_jsonl_path=BASELINE2_DIR / "private_debug.jsonl",
    submission_csv_path=BASELINE2_DIR / "submission.csv",
    report_json_path=BASELINE2_DIR / "private_report.json",
    show_progress=True,
)

pprint(baseline2_private_result.report)

In [ ]:
# At the end to download shit

!zip -r B2_results.zip /content/CSE151B_Kaggle/CSE151B_Kaggle/results/baseline2_prompt_format

from google.colab import files
files.download('B2_results.zip')

In [ ]:
"""
'generation_config': {'do_sample': True,
                       'max_new_tokens': 1024,
                       'min_p': 0.0,
                       'presence_penalty': 0.0,
                       'repetition_penalty': 1.0,
                       'temperature': 0.6,
                       'top_k': 20,
                       'top_p': 0.95},
 'output_jsonl_path': '/content/CSE151B_Kaggle/CSE151B_Kaggle/results/baseline2_prompt_format/private_results.jsonl',
 'problem_set': {'n': 943,
                 'n_answered': 0,
                 'n_free_form': 643,
                 'n_mcq': 300,
                 'name': 'private'},
 'schema_accuracy': {'retry_used_accuracy': None,
                     'retry_used_n': 0,
                     'schema_invalid_accuracy': None,
                     'schema_invalid_n': 0,
                     'schema_valid_accuracy': None,
                     'schema_valid_n': 0},
 'score_available': False,
 'submission_csv_path': '/content/CSE151B_Kaggle/CSE151B_Kaggle/results/baseline2_prompt_format/submission.csv',
 'summary': {'free_form_acc': None,
             'mcq_acc': None,
             'n_correct': 0,
             'n_free_form': 0,
             'n_mcq': 0,
             'n_scored': 0,
             'overall_acc': None},
 'timings': {'generation_sec': 334.20872846600014,
             'prompt_build_sec': 0.07997702800003026,
             'retry_generation_sec': 279.99592230300004,
             'scoring_sec': 0.004742979999718955}}
"""

In [ ]:
"""
'generation_config': {'do_sample': True,
                       'max_new_tokens': 4096,
                       'min_p': 0.0,
                       'presence_penalty': 0.0,
                       'repetition_penalty': 1.0,
                       'temperature': 0.6,
                       'top_k': 20,
                       'top_p': 0.95},
 'output_jsonl_path': '/content/CSE151B_Kaggle/CSE151B_Kaggle/results/baseline2_prompt_format/private_results.jsonl',
 'problem_set': {'n': 943,
                 'n_answered': 0,
                 'n_free_form': 643,
                 'n_mcq': 300,
                 'name': 'private'},
 'retry_generation_config': {'do_sample': False,
                             'max_new_tokens': 1024,
                             'min_p': 0.0,
                             'presence_penalty': 0.0,
                             'repetition_penalty': 1.0,
                             'temperature': 0.0,
                             'top_k': -1,
                             'top_p': 1.0},
 'schema_accuracy': {'retry_used_accuracy': None,
                     'retry_used_n': 0,
                     'sanitized_accuracy': None,
                     'sanitized_n': 0,
                     'schema_invalid_accuracy': None,
                     'schema_invalid_n': 0,
                     'schema_valid_accuracy': None,
                     'schema_valid_n': 0},
 'score_available': False,
 'submission_csv_path': '/content/CSE151B_Kaggle/CSE151B_Kaggle/results/baseline2_prompt_format/submission.csv',
 'summary': {'free_form_acc': None,
             'mcq_acc': None,
             'n_correct': 0,
             'n_free_form': 0,
             'n_mcq': 0,
             'n_scored': 0,
             'overall_acc': None},
 'timings': {'generation_sec': 1360.692972251,
             'prompt_build_sec': 0.08002269100006743,
             'retry_generation_sec': 79.66110282999989,
             'scoring_sec': 0.008488914999816188}}
"""

In [ ]:
"""
'generation_config': {'do_sample': True,
                       'max_new_tokens': 32768,
                       'min_p': 0.0,
                       'presence_penalty': 0.0,
                       'repetition_penalty': 1.0,
                       'temperature': 0.6,
                       'top_k': 20,
                       'top_p': 0.95},
 'output_jsonl_path': '/content/CSE151B_Kaggle/CSE151B_Kaggle/results/baseline2_prompt_format/private_results.jsonl',
 'problem_set': {'n': 943,
                 'n_answered': 0,
                 'n_free_form': 643,
                 'n_mcq': 300,
                 'name': 'private'},
 'retry_generation_config': {'do_sample': False,
                             'max_new_tokens': 32768,
                             'min_p': 0.0,
                             'presence_penalty': 0.0,
                             'repetition_penalty': 1.0,
                             'temperature': 0.0,
                             'top_k': -1,
                             'top_p': 1.0},
 'schema_accuracy': {'retry_used_accuracy': None,
                     'retry_used_n': 0,
                     'sanitized_accuracy': None,
                     'sanitized_n': 0,
                     'schema_invalid_accuracy': None,
                     'schema_invalid_n': 0,
                     'schema_valid_accuracy': None,
                     'schema_valid_n': 0},
 'score_available': False,
 'submission_csv_path': '/content/CSE151B_Kaggle/CSE151B_Kaggle/results/baseline2_prompt_format/submission.csv',
 'summary': {'free_form_acc': None,
             'mcq_acc': None,
             'n_correct': 0,
             'n_free_form': 0,
             'n_mcq': 0,
             'n_scored': 0,
             'overall_acc': None},
 'timings': {'generation_sec': 5990.060841825001,
             'prompt_build_sec': 0.5626943770002981,
             'retry_generation_sec': 307.0900928789997,
             'scoring_sec': 0.005678710997017333}}
  """

In [ ]:
pd.DataFrame([
    baseline2_val_result.report["formatting"]
]).T.rename(columns={0: "value"})

In [ ]:
baseline2_val_result.report["formatting"]["schema_error_counts"]

In [ ]:
bad_schema_rows = [
    row for row in baseline2_val_result.scored_rows
    if not row.get("schema_valid")
]

len(bad_schema_rows), bad_schema_rows[:3]

In [ ]:
comparison_rows = []

if "baseline1_val_result" in globals():
    comparison_rows.append({
        "baseline": "baseline1",
        **baseline1_val_result.report["summary"],
    })

comparison_rows.append({
    "baseline": "baseline2",
    **baseline2_val_result.report["summary"],
    "schema_valid_rate": baseline2_val_result.report["formatting"]["schema_valid_rate"],
    "extractable_rate": baseline2_val_result.report["formatting"]["extractable_rate"],
    "retry_rate": baseline2_val_result.report["formatting"]["retry_rate"],
})

pd.DataFrame(comparison_rows)

# Baseline 3

In [8]:
from baseline.baseline2_runner import run_baseline2_problem_set, run_baseline3_problem_set
from baseline.category_tagging import tag_problem_set_with_qwen, category_distribution
from baseline.experiments import append_comparison_row, build_comparison_row
from prompting.strategies import build_default_strategy_registry

BASELINE3_DIR = RESULTS_DIR / "baseline3_qwen_category_prompts"
BASELINE3_DIR.mkdir(parents=True, exist_ok=True)

PROMPT_STRATEGY_DIR = RESULTS_DIR / "prompt_strategy_experiments"
PROMPT_STRATEGY_DIR.mkdir(parents=True, exist_ok=True)

CATEGORY_EXPERIMENT_DIR = RESULTS_DIR / "baseline3_category_experiments"
CATEGORY_EXPERIMENT_DIR.mkdir(parents=True, exist_ok=True)

CATEGORY_EXPERIMENT_COMPARISON_CSV = RESULTS_DIR / "baseline3_category_experiment_comparison.csv"

EXPERIMENT_COMPARISON_CSV = RESULTS_DIR / "experiment_comparison.csv"

BASELINE3_TAGGING_DIR = RESULTS_DIR / "baseline3_category_tagging"
BASELINE3_TAGGING_DIR.mkdir(parents=True, exist_ok=True)

baseline3_generation_config = GenerationConfig(
    max_new_tokens=MAX_TOKENS,
    temperature=0.6,
    top_p=0.95,
    top_k=20,
    min_p=0.0,
    repetition_penalty=1.0,
    presence_penalty=0.0,
    do_sample=True,
)

baseline3_retry_generation_config = GenerationConfig(
    max_new_tokens=MAX_TOKENS,
    temperature=0.0,
    top_p=1.0,
    top_k=-1,
    min_p=0.0,
    repetition_penalty=1.0,
    presence_penalty=0.0,
    do_sample=False,
)

category_tag_generation_config = GenerationConfig(
    max_new_tokens=MAX_TOKENS,
    temperature=0.0,
    top_p=1.0,
    top_k=-1,
    min_p=0.0,
    repetition_penalty=1.0,
    presence_penalty=0.0,
    do_sample=False,
)

print("Tagging output dir:", BASELINE3_TAGGING_DIR)

print("Baseline 3 output dir:", BASELINE3_DIR)
print("Comparison CSV:", EXPERIMENT_COMPARISON_CSV)

Tagging output dir: /content/CSE151B_Kaggle/CSE151B_Kaggle/results/baseline3_category_tagging
Baseline 3 output dir: /content/CSE151B_Kaggle/CSE151B_Kaggle/results/baseline3_qwen_category_prompts
Comparison CSV: /content/CSE151B_Kaggle/CSE151B_Kaggle/results/experiment_comparison.csv


In [9]:
strategy_registry = build_default_strategy_registry()
strategy_rows = []

for strategy_name in strategy_registry.names():
    strategy = strategy_registry.get(strategy_name)
    for route in strategy.routes:
        strategy_rows.append({
            "strategy": strategy.name,
            "label": strategy.label,
            "route": route.name,
            "answer_format": route.answer_format,
            "category": route.category,
            "template_name": route.template_name or route.name,
        })

strategy_df = pd.DataFrame(strategy_rows)
display(strategy_df)
print("Strategies:", strategy_registry.names())

,strategy,label,route,answer_format,category,template_name
0,applied_word_problem_financial,applied_word_problem_financial,applied_word_problem_mcq,mcq,applied_word_problem,applied_word_problem_financial_applied_word_pr...
1,applied_word_problem_financial,applied_word_problem_financial,applied_word_problem_free_form,free_form,applied_word_problem,applied_word_problem_financial_applied_word_pr...
2,applied_word_problem_multi_step,applied_word_problem_multi_step,applied_word_problem_mcq,mcq,applied_word_problem,applied_word_problem_multi_step_applied_word_p...
3,applied_word_problem_multi_step,applied_word_problem_multi_step,applied_word_problem_free_form,free_form,applied_word_problem,applied_word_problem_multi_step_applied_word_p...
4,applied_word_problem_piecewise,applied_word_problem_piecewise,applied_word_problem_mcq,mcq,applied_word_problem,applied_word_problem_piecewise_applied_word_pr...
...,...,...,...,...,...,...
91,linear_algebra_lp_systems,linear_algebra_lp_systems,linear_algebra_free_form,free_form,linear_algebra,linear_algebra_lp_systems_linear_algebra_free_...
92,linear_algebra_mcq_option_verifier,linear_algebra_mcq_option_verifier,linear_algebra_mcq,mcq,linear_algebra,linear_algebra_mcq_option_verifier_linear_alge...
93,linear_algebra_mcq_option_verifier,linear_algebra_mcq_option_verifier,linear_algebra_free_form,free_form,linear_algebra,linear_algebra_mcq_option_verifier_linear_alge...
94,linear_algebra_v1_structured_verify,linear_algebra_structured_verify,linear_algebra_mcq,mcq,linear_algebra,linear_algebra_v1_structured_verify_linear_alg...


Strategies: ['applied_word_problem_financial', 'applied_word_problem_multi_step', 'applied_word_problem_piecewise', 'applied_word_problem_rate_distance', 'applied_word_problem_v2_model_building', 'arithmetic_algebra_mcq', 'arithmetic_algebra_multi_answer', 'arithmetic_algebra_numeric', 'arithmetic_algebra_symbolic', 'arithmetic_algebra_v2_general', 'baseline', 'baseline2', 'baseline3', 'baseline3_adaptive_rules', 'calculus_derivative_extrema', 'calculus_differential_equation', 'calculus_integral', 'calculus_limit_asymptotic', 'calculus_v1_structured', 'discrete_algorithm_v1_subtype_router', 'discrete_boolean_logic_v1', 'discrete_counting_dp_v1', 'discrete_number_theory_v1', 'discrete_sequence_option_verifier', 'general_math_mcq_verifier', 'general_math_multi_answer', 'general_math_text_or_unit', 'general_math_v1_structured', 'geometry_trig_frq', 'linear_algebra_freeform_multi_answer', 'linear_algebra_lp_systems', 'linear_algebra_mcq_option_verifier', 'linear_algebra_v1_structured_verif

## Question category tagging

In [ ]:
category_tag_generation_config.max_new_tokens

In [ ]:
BASELINE3_TAG_SMOKE_LIMIT = 8

tagged_val_smoke, val_tag_smoke_rows = tag_problem_set_with_qwen(
    problem_set=val_set,
    model_bundle=model_bundle,
    generation_config=category_tag_generation_config,
    batch_size=BATCH_SIZE,
    output_jsonl_path=BASELINE3_TAGGING_DIR / "val_category_tags_smoke.jsonl",
    output_one_hot_csv_path=BASELINE3_TAGGING_DIR / "val_category_one_hot_smoke.csv",
    limit=BASELINE3_TAG_SMOKE_LIMIT,
    show_progress=True,
)

smoke_df = pd.DataFrame([{
    "id": row["id"],
    "primary_category": row["primary_category"],
    "qwen_categories": row["qwen_categories"],
    "parse_ok": row["category_tag_parse_ok"],
    "source": row.get("category_tag_source"),
    "raw_head": row["category_tag_raw_output"][:180],
    "raw_tail": row["category_tag_raw_output"][-240:],
} for row in val_tag_smoke_rows])

display(smoke_df)

In [ ]:
for row in val_tag_smoke_rows:
    print("=" * 100)
    print("id:", row["id"])
    print("primary_category:", row["primary_category"])
    print("parse_ok:", row["category_tag_parse_ok"])
    print("source:", row.get("category_tag_source"))
    print(row["category_tag_raw_output"][:2000])

In [ ]:
category_cols = [
    "statistics_probability",
    "calculus",
    "geometry_trig",
    "linear_algebra",
    "discrete_algorithm",
    "arithmetic_algebra",
    "applied_word_problem",
    "general_math",
]

one_hot_smoke = pd.read_csv(BASELINE3_TAGGING_DIR / "val_category_one_hot_smoke.csv")
one_hot_smoke["num_categories"] = one_hot_smoke[category_cols].sum(axis=1)

display(one_hot_smoke[["id", "primary_category", "category_tag_source", "num_categories"]])
print(one_hot_smoke["num_categories"].value_counts())
print(one_hot_smoke["primary_category"].value_counts())

In [ ]:
tagged_public_set, public_category_rows = tag_problem_set_with_qwen(
    problem_set=public_set,
    model_bundle=model_bundle,
    generation_config=category_tag_generation_config,
    batch_size=BATCH_SIZE,
    output_jsonl_path=BASELINE3_TAGGING_DIR / "public_category_tags.jsonl",
    output_one_hot_csv_path=BASELINE3_TAGGING_DIR / "public_category_one_hot.csv",
    existing_tags_path=BASELINE3_TAGGING_DIR / "public_category_tags.jsonl",
    limit=None,
    show_progress=True,
)

print("Tagged public rows:", len(public_category_rows))
print(category_distribution(public_category_rows))

In [ ]:
public_one_hot = pd.read_csv(BASELINE3_TAGGING_DIR / "public_category_one_hot.csv")
public_one_hot["num_categories"] = public_one_hot[category_cols].sum(axis=1)

print("One-hot category count check:")
print(public_one_hot["num_categories"].value_counts())

print("\nPrimary category distribution:")
display(public_one_hot["primary_category"].value_counts().to_frame("count"))

print("\nTag source distribution:")
display(public_one_hot["category_tag_source"].value_counts().to_frame("count"))

In [ ]:
public_tags_df = pd.DataFrame(public_category_rows)

for category in category_cols:
    sample = public_tags_df[public_tags_df["primary_category"] == category].head(5)
    print("=" * 100)
    print("CATEGORY:", category, "n =", len(public_tags_df[public_tags_df["primary_category"] == category]))
    for _, row in sample.iterrows():
        print("-" * 80)
        print("id:", row["id"])
        print(str(row["question"])[:500])

In [ ]:
tagged_private_set, private_category_rows = tag_problem_set_with_qwen(
    problem_set=private_set,
    model_bundle=model_bundle,
    generation_config=category_tag_generation_config,
    batch_size=BATCH_SIZE,
    output_jsonl_path=BASELINE3_TAGGING_DIR / "private_category_tags.jsonl",
    output_one_hot_csv_path=BASELINE3_TAGGING_DIR / "private_category_one_hot.csv",
    existing_tags_path=BASELINE3_TAGGING_DIR / "private_category_tags.jsonl",
    limit=None,
    show_progress=True,
)

print("Tagged private rows:", len(private_category_rows))
print(category_distribution(private_category_rows))

In [ ]:
private_one_hot = pd.read_csv(BASELINE3_TAGGING_DIR / "private_category_one_hot.csv")
private_one_hot["num_categories"] = private_one_hot[category_cols].sum(axis=1)

print(private_one_hot["num_categories"].value_counts())
display(private_one_hot["primary_category"].value_counts().to_frame("count"))
display(private_one_hot["category_tag_source"].value_counts().to_frame("count"))

## Running category-specific experiments

In [10]:
import importlib

import baseline.rule_guidance
import baseline.category_rules
import baseline.category_harnesses
import baseline.baseline3_prompts
import prompting.strategies
import prompting.templates
import prompting.prompt_chain
import baseline.category_experiments

import baseline.category_work.calculus
import baseline.category_work.general_math
import baseline.category_work.arithmetic_algebra
import baseline.category_work.applied_word_problem
import baseline.category_work.geometry_trig
import baseline.category_work.statistics_probability

modules_to_reload = [
    baseline.rule_guidance,
    baseline.category_rules,
    baseline.category_harnesses,
    baseline.baseline3_prompts,
    prompting.strategies,
    prompting.templates,
    prompting.prompt_chain,
    baseline.category_experiments,
    baseline.category_work.calculus,
    baseline.category_work.general_math,
    baseline.category_work.arithmetic_algebra,
    baseline.category_work.applied_word_problem,
    baseline.category_work.geometry_trig,
    baseline.category_work.statistics_probability
]

for module in modules_to_reload:
    importlib.reload(module)

from baseline.rule_guidance import RULE_GUIDANCE_REGISTRY
from baseline.category_rules import (
    prepare_and_save_rule_annotations,
    rule_distribution,
)
from baseline.category_experiments import (
    load_tagged_problem_set,
    category_counts,
    category_summary_frame_rows,
    run_category_strategy_ablation,
    ablation_plan_frame,
    ablation_summary_frame,
    ablation_wrong_rows_frame,
    print_ablation_wrong_rows,
)

registered_category_work = {}

for module in [
    baseline.category_work.calculus,
    baseline.category_work.general_math,
    baseline.category_work.arithmetic_algebra,
    baseline.category_work.applied_word_problem,
    baseline.category_work.geometry_trig,
    baseline.category_work.statistics_probability
]:
    info = module.register_all()
    registered_category_work[info["category"]] = info

registered_guidance = baseline.rule_guidance.register_all_default_guidance()

print("registered_category_work:", registered_category_work.keys())
print("n_guidance:", len(RULE_GUIDANCE_REGISTRY._items))

registered_category_work: dict_keys(['calculus', 'general_math', 'arithmetic_algebra', 'applied_word_problem', 'geometry_trig', 'statistics_probability'])
n_guidance: 111


In [11]:
PUBLIC_DATA_PATH = "data/public.jsonl"

PUBLIC_CATEGORY_TAGS_PATH = (
    RESULTS_DIR
    / "baseline3_category_tagging"
    / "public_category_tags.jsonl"
)

tagged_public_set = load_tagged_problem_set(
    data_jsonl_path=PUBLIC_DATA_PATH,
    tags_jsonl_path=PUBLIC_CATEGORY_TAGS_PATH,
    name="public_tagged",
)

print(tagged_public_set.summary())
print(category_counts(tagged_public_set))

category_df = pd.DataFrame(category_summary_frame_rows(tagged_public_set)).sort_values(
    "n",
    ascending=False,
)

display(category_df)

{'name': 'public_tagged', 'n': 1126, 'n_mcq': 375, 'n_free_form': 751, 'n_answered': 1126}
{'applied_word_problem': 165, 'arithmetic_algebra': 358, 'calculus': 136, 'discrete_algorithm': 51, 'general_math': 10, 'geometry_trig': 136, 'linear_algebra': 30, 'statistics_probability': 240}


,category,n,share_of_dataset
1,arithmetic_algebra,358,0.317940
7,statistics_probability,240,0.213144
0,applied_word_problem,165,0.146536
2,calculus,136,0.120782
5,geometry_trig,136,0.120782
3,discrete_algorithm,51,0.045293
6,linear_algebra,30,0.026643
4,general_math,10,0.008881


In [12]:
RULE_ANNOTATION_DIR = RESULTS_DIR / "baseline3_rule_annotations"
RULE_ANNOTATION_DIR.mkdir(parents=True, exist_ok=True)

RULE_CATEGORIES = [
    "arithmetic_algebra",
    "statistics_probability",
    "applied_word_problem",
    "calculus",
    "geometry_trig",
    "general_math",
]

tagged_public_with_rules = prepare_and_save_rule_annotations(
    problem_set=tagged_public_set,
    categories=RULE_CATEGORIES,
    output_jsonl_path=RULE_ANNOTATION_DIR / "public_category_rules.jsonl",
    output_one_hot_csv_path=RULE_ANNOTATION_DIR / "public_category_rule_one_hot.csv",
    name="public_tagged_with_category_rules",
)

print(tagged_public_with_rules.summary())

smoke_rule_rows = [
    r for r in tagged_public_with_rules.records
    if r.get("primary_category") in RULE_CATEGORIES
]

print(rule_distribution(smoke_rule_rows))

{'name': 'public_tagged_with_category_rules', 'n': 1126, 'n_mcq': 375, 'n_free_form': 751, 'n_answered': 1126}
{'applied_word_problem_exact_expression_preferred': 64, 'applied_word_problem_exponential_modeling': 21, 'applied_word_problem_finance_percent_comparison': 32, 'applied_word_problem_function_interpretation': 9, 'applied_word_problem_geometry_application': 11, 'applied_word_problem_half_life_decay_exact': 9, 'applied_word_problem_inequality_constraint': 22, 'applied_word_problem_linear_modeling': 16, 'applied_word_problem_mcq': 21, 'applied_word_problem_multi_answer': 68, 'applied_word_problem_multi_step': 22, 'applied_word_problem_piecewise_case': 61, 'applied_word_problem_quantity_tracking': 51, 'applied_word_problem_rate_ratio_model': 35, 'applied_word_problem_rational_function_model': 2, 'applied_word_problem_step_function_ceiling': 4, 'applied_word_problem_table_schedule_reasoning': 12, 'applied_word_problem_unit_conversion_application': 10, 'arithmetic_algebra_base_arithm

In [13]:
rule_rows = []

for record in tagged_public_with_rules.records:
    if record.get("primary_category") not in RULE_CATEGORIES:
        continue

    rule_rows.append({
        "id": record.get("id"),
        "category": record.get("primary_category"),
        "is_mcq": bool(record.get("options")),
        "derived_rules": record.get("derived_rules"),
        "question": str(record.get("question", ""))[:250],
    })

rule_df = pd.DataFrame(rule_rows)

display(rule_df.head(20))
display(rule_df.explode("derived_rules")["derived_rules"].value_counts().to_frame("count"))

,id,category,is_mcq,derived_rules,question
0,0,arithmetic_algebra,False,[arithmetic_algebra_numeric_evaluation],Find the sum of the first $325$ positive even ...
1,1,calculus,True,[calculus_mcq_option_mapping],$int_{-infty}^{+infty} frac{a^{3/2}}{s^2+a^2} ...
2,2,calculus,False,[],A roasted turkey is taken from an oven when it...
3,3,arithmetic_algebra,False,[arithmetic_algebra_symbolic_manipulation],Reduce the fraction ${\frac{25}{40}}$. [ANS]
4,5,arithmetic_algebra,False,"[arithmetic_algebra_multi_answer, arithmetic_a...",An unscented beeswax candle melts at 145 $^\ci...
5,6,applied_word_problem,False,[applied_word_problem_exact_expression_preferr...,Let $p$ be the price of an item and $q$ be the...
6,7,arithmetic_algebra,False,[],The equation of the line that goes through the...
7,8,applied_word_problem,False,[applied_word_problem_exact_expression_preferr...,"In the early 1960s, radioactive strontium-90 w..."
8,9,applied_word_problem,True,"[applied_word_problem_geometry_application, ap...",The melting speed of half of the spherical sno...
9,10,arithmetic_algebra,True,"[arithmetic_algebra_conversion_representation,...",Let $k$ be the value of\n$$$\lfloor log_2 1 \r...


,count
derived_rules,
statistics_probability_hypothesis_test,182
statistics_probability_multi_answer,153
arithmetic_algebra_multi_answer,136
calculus_mcq_option_mapping,125
arithmetic_algebra_numeric_evaluation,124
...,...
calculus_complex_residue,1
calculus_equation_root_dichotomy,1
statistics_probability_chi_square_goodness_fit,1


In [14]:
all_registered_rule_names = sorted({
    rule.name
    for rule in baseline.category_rules.RULE_REGISTRY.all_rules()
    if rule.category in RULE_CATEGORIES
})

guidance_names = set(RULE_GUIDANCE_REGISTRY._items.keys())

missing_guidance = [
    name for name in all_registered_rule_names
    if name not in guidance_names
]

print("n_smoke_rules:", len(all_registered_rule_names))
print("n_guidance:", len(guidance_names))
print("missing_guidance:", missing_guidance)

n_smoke_rules: 90
n_guidance: 111
missing_guidance: []


In [15]:
from prompting.prompt_chain import build_prompt_chain
from prompting.models import problem_from_record

chain = build_prompt_chain(strategy_name="baseline3_adaptive_rules")

for category in RULE_CATEGORIES:
    candidates = [
        r for r in tagged_public_with_rules.records
        if r.get("primary_category") == category and r.get("derived_rules")
    ]

    if not candidates:
        print("=" * 100)
        print(category, "NO DERIVED RULES FOUND")
        continue

    sample = candidates[0]
    spec = chain.build_spec(problem_from_record(sample))
    user_msg = spec.to_messages()[-1]["content"]

    print("=" * 100)
    print(category, "id:", sample["id"])
    print("rules:", sample.get("derived_rules"))
    print("template:", spec.name)
    print("has subtype guidance:", "Subtype-specific guidance" in user_msg)
    print(user_msg[:900])

arithmetic_algebra id: 0
rules: ['arithmetic_algebra_numeric_evaluation']
template: baseline3_adaptive_rules_arithmetic_algebra_free_form
has subtype guidance: True
Subtype-specific guidance based on detected derived rules:
- Numeric evaluation: Compute carefully with order of operations, signs, fractions, exponents, and radicals. Keep exact form when useful and apply rounding only if requested.

Adaptive-rule instructions:
- Use the subtype-specific guidance above only when relevant.
- First count [ANS] blanks and identify what each blank asks for.
- Preserve exact forms unless a decimal is explicitly required.
- If a decimal is needed, provide extra precision when possible.
- For MCQ, always map the result to exactly one option letter.
- The final answer format requirements still take priority.

Find the sum of the first $325$ positive even whole numbers. Sum: [ANS]

Answer contract:
- Number of [ANS] blanks: 1
- Final boxed answer must contain the single requested answer.
- End with

In [16]:
rule_rows = [
    r for r in tagged_public_with_rules.records
    if r.get("primary_category") in RULE_CATEGORIES
]

dist = rule_distribution(rule_rows)

for key, value in sorted(dist.items()):
    print(f"{key}: {value}")

applied_word_problem_exact_expression_preferred: 64
applied_word_problem_exponential_modeling: 21
applied_word_problem_finance_percent_comparison: 32
applied_word_problem_function_interpretation: 9
applied_word_problem_geometry_application: 11
applied_word_problem_half_life_decay_exact: 9
applied_word_problem_inequality_constraint: 22
applied_word_problem_linear_modeling: 16
applied_word_problem_mcq: 21
applied_word_problem_multi_answer: 68
applied_word_problem_multi_step: 22
applied_word_problem_piecewise_case: 61
applied_word_problem_quantity_tracking: 51
applied_word_problem_rate_ratio_model: 35
applied_word_problem_rational_function_model: 2
applied_word_problem_step_function_ceiling: 4
applied_word_problem_table_schedule_reasoning: 12
applied_word_problem_unit_conversion_application: 10
arithmetic_algebra_base_arithmetic: 7
arithmetic_algebra_bernstein_polynomial: 1
arithmetic_algebra_conversion_representation: 33
arithmetic_algebra_discrete_integer: 59
arithmetic_algebra_equation

In [17]:
ABLATION_CATEGORIES = RULE_CATEGORIES

ABLATION_STRATEGIES = [
    {"name": "baseline3", "categories": "all", "label": "category_guided_baseline"},
    {"name": "baseline3_adaptive_rules", "categories": "all", "label": "category_plus_derived_rules"},
]

plan_df, skipped_df = ablation_plan_frame(
    ABLATION_CATEGORIES,
    ABLATION_STRATEGIES,
)

print("Will run:")
display(plan_df)

print("Will skip:")
display(skipped_df)

Will run:


,category,strategy_name,strategy_label,applicability_reason,strategy_notes
0,arithmetic_algebra,baseline3,category_guided_baseline,explicit_all,
1,arithmetic_algebra,baseline3_adaptive_rules,category_plus_derived_rules,explicit_all,
2,statistics_probability,baseline3,category_guided_baseline,explicit_all,
3,statistics_probability,baseline3_adaptive_rules,category_plus_derived_rules,explicit_all,
4,applied_word_problem,baseline3,category_guided_baseline,explicit_all,
5,applied_word_problem,baseline3_adaptive_rules,category_plus_derived_rules,explicit_all,
6,calculus,baseline3,category_guided_baseline,explicit_all,
7,calculus,baseline3_adaptive_rules,category_plus_derived_rules,explicit_all,
8,geometry_trig,baseline3,category_guided_baseline,explicit_all,
9,geometry_trig,baseline3_adaptive_rules,category_plus_derived_rules,explicit_all,


Will skip:


""


In [18]:
SMOKE_LIMIT_PER_COMBO = 32

smoke_ablation = run_category_strategy_ablation(
    problem_set=tagged_public_with_rules,
    categories=ABLATION_CATEGORIES,
    strategies=ABLATION_STRATEGIES,
    model_bundle=model_bundle,
    generation_config=baseline3_generation_config,
    retry_generation_config=baseline3_retry_generation_config,
    experiment_name="category_adaptive_rules_smoke",
    batch_size=BATCH_SIZE,
    limit_per_combo=SMOKE_LIMIT_PER_COMBO,
    score=True,
    output_dir=CATEGORY_EXPERIMENT_DIR,
    comparison_csv_path=None,
    show_progress=True,
)

smoke_summary_df = ablation_summary_frame(smoke_ablation)
display(smoke_summary_df.sort_values(["category", "strategy_name"]))
print(smoke_ablation["artifacts"])

Ablation experiment: category_adaptive_rules_smoke
Category: arithmetic_algebra
Strategy: baseline3
Applicability: explicit_all


Generating:   0%|          | 0/1 [00:00<?, ?it/s]

Rendering prompts:   0%|          | 0/32 [00:00<?, ?it/s]

Processed prompts:   0%|          | 0/32 [00:00<?, ?it/s, est. speed input: 0.00 toks/s, output: 0.00 toks/s]

Generating:   0%|          | 0/1 [00:00<?, ?it/s]

...

Generating:   0%|          | 0/1 [00:00<?, ?it/s]

Rendering prompts:   0%|          | 0/2 [00:00<?, ?it/s]

Processed prompts:   0%|          | 0/2 [00:00<?, ?it/s, est. speed input: 0.00 toks/s, output: 0.00 toks/s]

,experiment_name,category,strategy_name,applicability_reason,n_scored,n_correct,n_errors,overall_acc,mcq_acc,free_form_acc,...,retry_rate,retry_used_rate,harness_avg_pass_rate,harness_full_pass_rate,output_jsonl_path,debug_jsonl_path,report_json_path,wrong_rows_jsonl_path,wrong_rows_txt_path,share_of_ablation_errors
4,category_adaptive_rules_smoke,applied_word_problem,baseline3,explicit_all,32,21,11,0.65625,0.800000,0.629630,...,0.00000,0.00000,0.841003,0.31250,/content/CSE151B_Kaggle/CSE151B_Kaggle/results...,/content/CSE151B_Kaggle/CSE151B_Kaggle/results...,None,/content/CSE151B_Kaggle/CSE151B_Kaggle/results...,/content/CSE151B_Kaggle/CSE151B_Kaggle/results...,0.073826
5,category_adaptive_rules_smoke,applied_word_problem,baseline3_adaptive_rules,explicit_all,32,20,12,0.62500,0.600000,0.629630,...,0.06250,0.00000,0.794073,0.28125,/content/CSE151B_Kaggle/CSE151B_Kaggle/results...,/content/CSE151B_Kaggle/CSE151B_Kaggle/results...,None,/content/CSE151B_Kaggle/CSE151B_Kaggle/results...,/content/CSE151B_Kaggle/CSE151B_Kaggle/results...,0.080537
0,category_adaptive_rules_smoke,arithmetic_algebra,baseline3,explicit_all,32,17,15,0.53125,0.625000,0.500000,...,0.09375,0.00000,0.796209,0.50000,/content/CSE151B_Kaggle/CSE151B_Kaggle/results...,/content/CSE151B_Kaggle/CSE151B_Kaggle/results...,None,/content/CSE151B_Kaggle/CSE151B_Kaggle/results...,/content/CSE151B_Kaggle/CSE151B_Kaggle/results...,0.100671
1,category_adaptive_rules_smoke,arithmetic_algebra,baseline3_adaptive_rules,explicit_all,32,18,14,0.56250,0.625000,0.541667,...,0.09375,0.00000,0.801241,0.53125,/content/CSE151B_Kaggle/CSE151B_Kaggle/results...,/content/CSE151B_Kaggle/CSE151B_Kaggle/results...,None,/content/CSE151B_Kaggle/CSE151B_Kaggle/results...,/content/CSE151B_Kaggle/CSE151B_Kaggle/results...,0.093960
6,category_adaptive_rules_smoke,calculus,baseline3,explicit_all,32,16,16,0.50000,0.551724,0.000000,...,0.03125,0.03125,0.790031,0.09375,/content/CSE151B_Kaggle/CSE151B_Kaggle/results...,/content/CSE151B_Kaggle/CSE151B_Kaggle/results...,None,/content/CSE151B_Kaggle/CSE151B_Kaggle/results...,/content/CSE151B_Kaggle/CSE151B_Kaggle/results...,0.107383
7,category_adaptive_rules_smoke,calculus,baseline3_adaptive_rules,explicit_all,32,22,10,0.68750,0.758621,0.000000,...,0.00000,0.00000,0.837715,0.12500,/content/CSE151B_Kaggle/CSE151B_Kaggle/results...,/content/CSE151B_Kaggle/CSE151B_Kaggle/results...,None,/content/CSE151B_Kaggle/CSE151B_Kaggle/results...,/content/CSE151B_Kaggle/CSE151B_Kaggle/results...,0.067114
10,category_adaptive_rules_smoke,general_math,baseline3,explicit_all,10,2,8,0.20000,0.250000,0.000000,...,0.20000,0.10000,0.698119,0.20000,/content/CSE151B_Kaggle/CSE151B_Kaggle/results...,/content/CSE151B_Kaggle/CSE151B_Kaggle/results...,None,/content/CSE151B_Kaggle/CSE151B_Kaggle/results...,/content/CSE151B_Kaggle/CSE151B_Kaggle/results...,0.053691
11,category_adaptive_rules_smoke,general_math,baseline3_adaptive_rules,explicit_all,10,2,8,0.20000,0.250000,0.000000,...,0.20000,0.00000,0.634483,0.20000,/content/CSE151B_Kaggle/CSE151B_Kaggle/results...,/content/CSE151B_Kaggle/CSE151B_Kaggle/results...,None,/content/CSE151B_Kaggle/CSE151B_Kaggle/results...,/content/CSE151B_Kaggle/CSE151B_Kaggle/results...,0.053691
8,category_adaptive_rules_smoke,geometry_trig,baseline3,explicit_all,32,17,15,0.53125,0.800000,0.409091,...,0.09375,0.03125,0.736183,0.28125,/content/CSE151B_Kaggle/CSE151B_Kaggle/results...,/content/CSE151B_Kaggle/CSE151B_Kaggle/results...,None,/content/CSE151B_Kaggle/CSE151B_Kaggle/results...,/content/CSE151B_Kaggle/CSE151B_Kaggle/results...,0.100671
9,category_adaptive_rules_smoke,geometry_trig,baseline3_adaptive_rules,explicit_all,32,21,11,0.65625,0.900000,0.545455,...,0.03125,0.00000,0.808056,0.31250,/content/CSE151B_Kaggle/CSE151B_Kaggle/results...,/content/CSE151B_Kaggle/CSE151B_Kaggle/results...,None,/content/CSE151B_Kaggle/CSE151B_Kaggle/results...,/content/CSE151B_Kaggle/CSE151B_Kaggle/results...,0.073826


{'ablation_dir': '/content/CSE151B_Kaggle/CSE151B_Kaggle/results/baseline3_category_experiments/_ablation/category_adaptive_rules_smoke', 'summary_csv_path': '/content/CSE151B_Kaggle/CSE151B_Kaggle/results/baseline3_category_experiments/_ablation/category_adaptive_rules_smoke/ablation_summary.csv', 'skipped_csv_path': '/content/CSE151B_Kaggle/CSE151B_Kaggle/results/baseline3_category_experiments/_ablation/category_adaptive_rules_smoke/ablation_skipped.csv', 'summary_json_path': '/content/CSE151B_Kaggle/CSE151B_Kaggle/results/baseline3_category_experiments/_ablation/category_adaptive_rules_smoke/ablation_summary.json'}


In [19]:
pivot = smoke_summary_df.pivot_table(
    index="category",
    columns="strategy_name",
    values="overall_acc",
    aggfunc="first",
)

pivot["delta_adaptive_minus_baseline3"] = (
    pivot["baseline3_adaptive_rules"] - pivot["baseline3"]
)

display(
    pivot.sort_values(
        "delta_adaptive_minus_baseline3",
        ascending=False,
    )
)

strategy_name,baseline3,baseline3_adaptive_rules,delta_adaptive_minus_baseline3
category,,,
calculus,0.50000,0.68750,0.18750
statistics_probability,0.46875,0.62500,0.15625
geometry_trig,0.53125,0.65625,0.12500
arithmetic_algebra,0.53125,0.56250,0.03125
general_math,0.20000,0.20000,0.00000
applied_word_problem,0.65625,0.62500,-0.03125


In [20]:
wrong_smoke_df = ablation_wrong_rows_frame(
    smoke_ablation,
    category="arithmetic_algebra",
    strategy_name="baseline3_adaptive_rules",
    max_rows=10,
)

display(wrong_smoke_df[
    [
        "id",
        "is_mcq",
        "gold",
        "boxed_answer",
        "correct",
        "schema_valid",
        "question",
        "response_tail",
    ]
])

,id,is_mcq,gold,boxed_answer,correct,schema_valid,question,response_tail
0,5,False,"[62.7777777777778, 335.927777777778, 604.67]","62.7778, 335.9278, 604.67",False,True,An unscented beeswax candle melts at 145 $^\ci...,Reasoning:\nAnswer extracted from completed fi...
1,21,False,"[3*t^1*(1-t)^2, 6*t^2*(1-t)^2, 10*t^3*(1-t)^2,...","3t(1-t)^2,6t^2(1-t)^2,10t^3(1-t)^2,7t^6(1-t),7...",False,True,Write the formula for the 1st Bernstein polyno...,Reasoning:\nAnswer extracted from completed fi...
2,37,False,"[110101, 11010101, 1010100001]",,False,False,Add the following binary numbers $\begin{array...,to bit8):\n\ncarry = 0\n\nbit0: 1 + 0 + 0 = 1 ...
3,39,False,[2.2892],2.3,False,True,Solve $p=30 (0.8)^q$ graphically for $q$ if $p...,Reasoning:\nAnswer extracted from completed fi...
4,41,False,"[T * W + S*(T+W), T+W, 65.2389937106918]",,False,False,"The total resistance, $R$, of a particular gro...",vide the answers as per the problem's requirem...
5,43,False,"[YES, -6.70156211871642, -0.298437881283576]","Yes, (-7 - \sqrt{41})/2, (-7 + \sqrt{41})/2",False,True,Find all real solutions of equation $2+7 z+z^2...,Reasoning:\nAnswer extracted from completed fi...
6,53,True,C,H,False,True,"Let \( n \) be any positive integer, and alpha...",Reasoning:\nAnswer extracted from completed fi...
7,55,False,"[6*e^(16*x), 6]","6e^{16x},6",False,True,Let $u(x)=e^{8x}$ and $v(x)=6x+6$. Find a simp...,Reasoning:\nAnswer extracted from completed fi...
8,61,False,[BCEG],"B,C,E,G",False,True,How does the symmetry of $ f(x)=\frac{p(x)}{q(...,Reasoning:\nAnswer extracted from completed fi...
9,62,False,"[(2, -2)]","-2,2",False,True,Given that $x=1$ is a zero of the polynomial $...,Reasoning:\nAnswer extracted from completed fi...


In [21]:
print_ablation_wrong_rows(
    smoke_ablation,
    category="arithmetic_algebra",
    strategy_name="baseline3_adaptive_rules",
    max_rows=5,
    max_question_chars=1200,
    max_response_chars=2500,
)

WRONG ROW 1
id: 5
category: arithmetic_algebra
route_name: arithmetic_algebra_free_form
template_name: baseline3_adaptive_rules_arithmetic_algebra_free_form
is_mcq: False
options: None
gold: ['62.7777777777778', '335.927777777778', '604.67']
boxed_answer: 62.7778, 335.9278, 604.67
extracted_final_answer: 62.7778, 335.9278, 604.67
correct: False
schema_valid: True
schema_errors: []
harness_pass_rate: 0.7142857142857143
harness_check_results: [{'name': 'schema_valid', 'passed': True, 'weight': 1.0, 'details': {}}, {'name': 'extractable', 'passed': True, 'weight': 1.0, 'details': {}}, {'name': 'answer_count_valid', 'passed': True, 'weight': 1.0, 'details': {'expected': 3, 'actual': 3}}, {'name': 'mcq_letter_valid', 'passed': None, 'weight': 1.0, 'details': {}}, {'name': 'exact_form_preferred_unless_requested', 'passed': True, 'weight': 0.75, 'details': {'approximation_requested': False, 'suspicious_terms': []}}, {'name': 'symbolic_answer_not_over_decimalized', 'passed': None, 'weight': 0.

In [22]:
def result_map(ablation_result, category, strategy_name):
    result = (
        ablation_result
        .get("results", {})
        .get(category, {})
        .get(strategy_name)
    )
    if result is None:
        return {}

    return {
        row["id"]: row
        for row in result.scored_rows
    }


def compare_category_strategies(ablation_result, category):
    base = result_map(ablation_result, category, "baseline3")
    adaptive = result_map(ablation_result, category, "baseline3_adaptive_rules")

    rows = []

    for problem_id in sorted(set(base) | set(adaptive)):
        b = base.get(problem_id, {})
        a = adaptive.get(problem_id, {})

        b_correct = b.get("correct")
        a_correct = a.get("correct")

        if b_correct is True and a_correct is True:
            status = "both_correct"
        elif b_correct is False and a_correct is False:
            status = "both_wrong"
        elif b_correct is False and a_correct is True:
            status = "adaptive_win"
        elif b_correct is True and a_correct is False:
            status = "adaptive_loss"
        else:
            status = "unknown"

        rows.append({
            "id": problem_id,
            "category": category,
            "status": status,
            "baseline_correct": b_correct,
            "adaptive_correct": a_correct,
            "is_mcq": a.get("is_mcq", b.get("is_mcq")),
            "gold": a.get("gold", b.get("gold")),
            "baseline_boxed": b.get("boxed_answer"),
            "adaptive_boxed": a.get("boxed_answer"),
            "baseline_schema_valid": b.get("schema_valid"),
            "adaptive_schema_valid": a.get("schema_valid"),
            "question": str(a.get("question") or b.get("question") or "")[:500],
        })

    return pd.DataFrame(rows)


comparison_dfs = {}

for category in ABLATION_CATEGORIES:
    df = compare_category_strategies(smoke_ablation, category)
    comparison_dfs[category] = df

    print("=" * 100)
    print(category)
    display(df["status"].value_counts().to_frame("count"))
    display(df.sort_values(["status", "id"]).head(20))

arithmetic_algebra


,count
status,
both_correct,15
both_wrong,12
adaptive_win,3
adaptive_loss,2


,id,category,status,baseline_correct,adaptive_correct,is_mcq,gold,baseline_boxed,adaptive_boxed,baseline_schema_valid,adaptive_schema_valid,question
16,53,arithmetic_algebra,adaptive_loss,True,False,True,C,C,H,True,True,"Let \( n \) be any positive integer, and alpha..."
19,61,arithmetic_algebra,adaptive_loss,True,False,False,[BCEG],BCEG,"B,C,E,G",True,True,How does the symmetry of $ f(x)=\frac{p(x)}{q(...
22,67,arithmetic_algebra,adaptive_win,False,True,False,[122],148,122,True,True,A plumber and his assistant work together to r...
25,72,arithmetic_algebra,adaptive_win,False,True,False,"[190, 250]","200,250","190,250",True,True,Show how you can add and subtract mentally. Tr...
28,88,arithmetic_algebra,adaptive_win,False,True,True,D,,D,False,True,"Let $a, b$ be positive integers such that\n$$$..."
0,0,arithmetic_algebra,both_correct,True,True,False,[325*(1+325)],105950,105950,True,True,Find the sum of the first $325$ positive even ...
1,3,arithmetic_algebra,both_correct,True,True,False,[5/8],\dfrac{5}{8},\dfrac{5}{8},True,True,Reduce the fraction ${\frac{25}{40}}$. [ANS]
3,7,arithmetic_algebra,both_correct,True,True,False,[1.44444444444444],\dfrac{13}{9},\dfrac{13}{9},True,True,The equation of the line that goes through the...
4,10,arithmetic_algebra,both_correct,True,True,True,E,E,E,True,True,Let $k$ be the value of\n$$$\lfloor log_2 1 \r...
5,19,arithmetic_algebra,both_correct,True,True,True,E,E,E,True,True,A sequence of positive reals defined by $a_0=x...


statistics_probability


,count
status,
both_correct,15
both_wrong,12
adaptive_win,5


...

,count
status,
both_correct,17
both_wrong,11
adaptive_win,4


,id,category,status,baseline_correct,adaptive_correct,is_mcq,gold,baseline_boxed,adaptive_boxed,baseline_schema_valid,adaptive_schema_valid,question
5,36,geometry_trig,adaptive_win,False,True,False,"[2*pi/6, 0.7]","0.7,\dfrac{\pi}{3}","π/3, 0.7",True,True,Find the period and amplitude of $r=0.7 \sin(6...
13,75,geometry_trig,adaptive_win,False,True,False,"[13^2 + (x-4)^2 = x^2, 23.125]","x^2 = (x - 4)^2 + 169, 23.125","8x - 185 = 0, 23.125",True,True,A tree is supported by a wire anchored in the ...
22,151,geometry_trig,adaptive_win,False,True,False,"[0.564642473395035, 0.825335614909678, 0.68413...","0.5646424733,0.8253356149,0.6840900014","0.564642, 0.825336, 0.684137",True,True,Find the following values: $\sin (0.6)=$ [ANS]...
23,153,geometry_trig,adaptive_win,False,True,True,H,,H,False,True,"In $\triangle{ABC}$ , let $D$ and $E$ be point..."
0,11,geometry_trig,both_correct,True,True,True,G,G,G,True,True,"Let $ABC$ be a triangle with $AB = 13$ , $BC =..."
2,17,geometry_trig,both_correct,True,True,False,"[2*8*x, 2*8*x]","16x,16x","16x,16x",True,True,Use the formula for lowering the powers to sim...
3,31,geometry_trig,both_correct,True,True,False,"[K, I, J, F, B, H]","K,I,J,F,B,H","K,I,J,F,B,H",True,True,Match each given expression with one of the ex...
7,46,geometry_trig,both_correct,True,True,False,"[429.804, 1012.555]","429.804,1012.555","429.804, 1012.555",True,True,A bullet is fired into the air with an initial...
8,50,geometry_trig,both_correct,True,True,True,B,B,B,True,True,Calculate $E = \left(\sin\left(\frac{ \pi }{ 8...
9,59,geometry_trig,both_correct,True,True,True,B,B,B,True,True,Circle $\omega_1$ is defined by the equation $...


general_math


,count
status,
both_wrong,8
both_correct,2


,id,category,status,baseline_correct,adaptive_correct,is_mcq,gold,baseline_boxed,adaptive_boxed,baseline_schema_valid,adaptive_schema_valid,question
2,449,general_math,both_correct,True,True,True,G,G,G,True,True,Let $x$ be the arithmetic mean of all positive...
7,1035,general_math,both_correct,True,True,True,G,G,G,True,True,"Given \[ 11z^{10}+10iz^9+10iz-11=0, \] find th..."
0,95,general_math,both_wrong,False,False,True,H,C,C,True,True,"For positive integers $n$ , let $f(n)$ denote ..."
1,130,general_math,both_wrong,False,False,True,F,B,B,True,True,"In the coordinate plane, an ant begins at the ..."
3,560,general_math,both_wrong,False,False,True,B,A,A,True,True,"For each positive integer $n$, let $k(n)$ be t..."
4,623,general_math,both_wrong,False,False,True,A,C,C,True,True,Let $\mathbf{F}$ be the set $A = {\frac{0.1}{a...
5,839,general_math,both_wrong,False,False,False,998,,,False,False,Vaysha has a board with $999$ consecutive numb...
6,1032,general_math,both_wrong,False,False,True,B,A,A,True,True,Let $z$ be a complex number such that $z^7=1$ ...
8,1075,general_math,both_wrong,False,False,True,F,B,B,True,True,"For each positive integer $k$, let $A(k)$ be t..."
9,1118,general_math,both_wrong,False,False,False,2^{999}-2^{499},0,,True,False,"$S={1,2,...,1000}$ and $T'=\left\{ 1001-t|t \i..."


In [23]:
all_wrong_rows = []

for category in ABLATION_CATEGORIES:
    wrong_df = ablation_wrong_rows_frame(
        smoke_ablation,
        category=category,
        strategy_name="baseline3_adaptive_rules",
        max_rows=50,
        max_question_chars=800,
        max_response_chars=1800,
    )

    if len(wrong_df) > 0:
        wrong_df["category"] = category
        all_wrong_rows.append(wrong_df)

all_wrong_df = pd.concat(all_wrong_rows, ignore_index=True)

display(all_wrong_df[
    [
        "category",
        "id",
        "is_mcq",
        "gold",
        "boxed_answer",
        "correct",
        "schema_valid",
        "harness_pass_rate",
        "question",
        "response_tail",
    ]
])

,category,id,is_mcq,gold,boxed_answer,correct,schema_valid,harness_pass_rate,question,response_tail
0,arithmetic_algebra,5,False,"[62.7777777777778, 335.927777777778, 604.67]","62.7778, 335.9278, 604.67",False,True,0.714286,An unscented beeswax candle melts at 145 $^\ci...,Reasoning:\nAnswer extracted from completed fi...
1,arithmetic_algebra,21,False,"[3*t^1*(1-t)^2, 6*t^2*(1-t)^2, 10*t^3*(1-t)^2,...","3t(1-t)^2,6t^2(1-t)^2,10t^3(1-t)^2,7t^6(1-t),7...",False,True,0.724138,Write the formula for the 1st Bernstein polyno...,Reasoning:\nAnswer extracted from completed fi...
2,arithmetic_algebra,37,False,"[110101, 11010101, 1010100001]",,False,False,0.115385,Add the following binary numbers $\begin{array...,So the binary sum is 1 0 1 0 0 0 0 0 0 1 → wai...
3,arithmetic_algebra,39,False,[2.2892],2.3,False,True,0.545455,Solve $p=30 (0.8)^q$ graphically for $q$ if $p...,Reasoning:\nAnswer extracted from completed fi...
4,arithmetic_algebra,41,False,"[T * W + S*(T+W), T+W, 65.2389937106918]",,False,False,0.000000,"The total resistance, $R$, of a particular gro...","t A and B are expressions.\n\nWait, but the pr..."
...,...,...,...,...,...,...,...,...,...,...
62,general_math,623,True,A,C,False,True,0.724138,Let $\mathbf{F}$ be the set $A = {\frac{0.1}{a...,Reasoning:\nAnswer extracted from completed fi...
63,general_math,839,False,998,,False,False,0.000000,Vaysha has a board with $999$ consecutive numb...,99 minus the maximum number of i's that have a...
64,general_math,1032,True,B,A,False,True,0.724138,Let $z$ be a complex number such that $z^7=1$ ...,Reasoning:\nAnswer extracted from completed fi...
65,general_math,1075,True,F,B,False,True,0.724138,"For each positive integer $k$, let $A(k)$ be t...",Reasoning:\nAnswer extracted from completed fi...


In [24]:
display(
    all_wrong_df.groupby(["category", "is_mcq", "schema_valid"])
    .size()
    .to_frame("n")
    .reset_index()
    .sort_values(["category", "n"], ascending=[True, False])
)

,category,is_mcq,schema_valid,n
1,applied_word_problem,False,True,8
0,applied_word_problem,False,False,2
2,applied_word_problem,True,True,2
4,arithmetic_algebra,False,True,9
3,arithmetic_algebra,False,False,2
6,arithmetic_algebra,True,True,2
5,arithmetic_algebra,True,False,1
8,calculus,True,True,7
7,calculus,False,True,3
10,general_math,True,True,6


In [25]:
for category in ABLATION_CATEGORIES:
    print("=" * 120)
    print("CATEGORY:", category)
    print("=" * 120)

    print_ablation_wrong_rows(
        smoke_ablation,
        category=category,
        strategy_name="baseline3_adaptive_rules",
        max_rows=5,
        max_question_chars=1000,
        max_response_chars=2200,
    )

CATEGORY: arithmetic_algebra
WRONG ROW 1
id: 5
category: arithmetic_algebra
route_name: arithmetic_algebra_free_form
template_name: baseline3_adaptive_rules_arithmetic_algebra_free_form
is_mcq: False
options: None
gold: ['62.7777777777778', '335.927777777778', '604.67']
boxed_answer: 62.7778, 335.9278, 604.67
extracted_final_answer: 62.7778, 335.9278, 604.67
correct: False
schema_valid: True
schema_errors: []
harness_pass_rate: 0.7142857142857143
harness_check_results: [{'name': 'schema_valid', 'passed': True, 'weight': 1.0, 'details': {}}, {'name': 'extractable', 'passed': True, 'weight': 1.0, 'details': {}}, {'name': 'answer_count_valid', 'passed': True, 'weight': 1.0, 'details': {'expected': 3, 'actual': 3}}, {'name': 'mcq_letter_valid', 'passed': None, 'weight': 1.0, 'details': {}}, {'name': 'exact_form_preferred_unless_requested', 'passed': True, 'weight': 0.75, 'details': {'approximation_requested': False, 'suspicious_terms': []}}, {'name': 'symbolic_answer_not_over_decimalized',

In [26]:
ABLATION_CATEGORIES = ["calculus"]

In [27]:
full_ablation = run_category_strategy_ablation(
    problem_set=tagged_public_with_rules,
    categories=ABLATION_CATEGORIES,
    strategies=ABLATION_STRATEGIES,
    model_bundle=model_bundle,
    generation_config=baseline3_generation_config,
    retry_generation_config=baseline3_retry_generation_config,
    experiment_name="category_adaptive_rules_full",
    batch_size=BATCH_SIZE,
    limit_per_combo=None,
    score=True,
    output_dir=CATEGORY_EXPERIMENT_DIR,
    comparison_csv_path=None,
    show_progress=True,
)

full_summary_df = ablation_summary_frame(full_ablation)
display(
    full_summary_df.sort_values(
        ["share_of_ablation_errors", "category", "strategy_name"],
        ascending=[False, True, True],
    )
)

print(full_ablation["artifacts"])

Ablation experiment: category_adaptive_rules_full
Category: calculus
Strategy: baseline3
Applicability: explicit_all


Generating:   0%|          | 0/5 [00:00<?, ?it/s]

Rendering prompts:   0%|          | 0/32 [00:00<?, ?it/s]

Processed prompts:   0%|          | 0/32 [00:00<?, ?it/s, est. speed input: 0.00 toks/s, output: 0.00 toks/s]

Rendering prompts:   0%|          | 0/32 [00:00<?, ?it/s]

...

Generating:   0%|          | 0/1 [00:00<?, ?it/s]

Rendering prompts:   0%|          | 0/6 [00:00<?, ?it/s]

Processed prompts:   0%|          | 0/6 [00:00<?, ?it/s, est. speed input: 0.00 toks/s, output: 0.00 toks/s]

,experiment_name,category,strategy_name,applicability_reason,n_scored,n_correct,n_errors,overall_acc,mcq_acc,free_form_acc,...,retry_rate,retry_used_rate,harness_avg_pass_rate,harness_full_pass_rate,output_jsonl_path,debug_jsonl_path,report_json_path,wrong_rows_jsonl_path,wrong_rows_txt_path,share_of_ablation_errors
0,category_adaptive_rules_full,calculus,baseline3,explicit_all,136,94,42,0.691176,0.712,0.454545,...,0.036765,0.036765,0.854264,0.220588,/content/CSE151B_Kaggle/CSE151B_Kaggle/results...,/content/CSE151B_Kaggle/CSE151B_Kaggle/results...,None,/content/CSE151B_Kaggle/CSE151B_Kaggle/results...,/content/CSE151B_Kaggle/CSE151B_Kaggle/results...,0.545455
1,category_adaptive_rules_full,calculus,baseline3_adaptive_rules,explicit_all,136,101,35,0.742647,0.768,0.454545,...,0.044118,0.044118,0.867512,0.235294,/content/CSE151B_Kaggle/CSE151B_Kaggle/results...,/content/CSE151B_Kaggle/CSE151B_Kaggle/results...,None,/content/CSE151B_Kaggle/CSE151B_Kaggle/results...,/content/CSE151B_Kaggle/CSE151B_Kaggle/results...,0.454545


{'ablation_dir': '/content/CSE151B_Kaggle/CSE151B_Kaggle/results/baseline3_category_experiments/_ablation/category_adaptive_rules_full', 'summary_csv_path': '/content/CSE151B_Kaggle/CSE151B_Kaggle/results/baseline3_category_experiments/_ablation/category_adaptive_rules_full/ablation_summary.csv', 'skipped_csv_path': '/content/CSE151B_Kaggle/CSE151B_Kaggle/results/baseline3_category_experiments/_ablation/category_adaptive_rules_full/ablation_skipped.csv', 'summary_json_path': '/content/CSE151B_Kaggle/CSE151B_Kaggle/results/baseline3_category_experiments/_ablation/category_adaptive_rules_full/ablation_summary.json'}


In [30]:
wrong_df = ablation_wrong_rows_frame(
    full_ablation,
    category="calculus",
    strategy_name="baseline3_adaptive_rules",
    max_rows=30,
    max_question_chars=1000,
    max_response_chars=2500,
)

display(wrong_df[
    [
        "id",
        "is_mcq",
        "gold",
        "boxed_answer",
        "correct",
        "schema_valid",
        "harness_pass_rate",
        "question",
        "response_tail",
    ]
])

,id,is_mcq,gold,boxed_answer,correct,schema_valid,harness_pass_rate,question,response_tail
0,1,True,F,E,False,True,0.724138,$int_{-infty}^{+infty} frac{a^{3/2}}{s^2+a^2} ...,Reasoning:\nAnswer extracted from completed fi...
1,2,False,"[143.224229233795, 2.32624773420025]","143, 2.33",False,True,0.636364,A roasted turkey is taken from an oven when it...,Reasoning:\nAnswer extracted from completed fi...
2,48,True,I,E,False,True,0.656250,Integral $int_{0}^{4}{frac{sqrt{x}}{1+xsqrt{x}...,Reasoning:\nAnswer extracted from completed fi...
3,66,False,"[77.2*exp(0.016*t), 92.056, 2020]","77.2 e^{0.016 t}, 92.053, 2020",False,True,0.636364,World poultry production was 77.2 million tons...,Reasoning:\nAnswer extracted from completed fi...
4,90,True,F,A,False,True,0.656250,Find the derivative of the function $y = \frac...,Reasoning:\nAnswer extracted from completed fi...
5,96,True,B,J,False,True,0.656250,Compute the integral:\n$$\n\int \frac{ 2 \cdot...,Reasoning:\nAnswer extracted from completed fi...
6,113,False,[14.115024216],14.112,False,True,0.560000,The length of a cube was measured and found to...,Reasoning:\nAnswer extracted from completed fi...
7,253,True,F,H,False,True,0.656250,"Let $f \! \in\! L ( [ a, b ] )$, and define $g...",Reasoning:\nAnswer extracted from completed fi...
8,257,True,F,J,False,True,0.656250,Let $f(x)$ be absolutely continuous on any int...,Reasoning:\nAnswer extracted from completed fi...
9,280,True,E,C,False,True,0.656250,Compute the integral:\n$$\n\int \frac{ -\sin(2...,Reasoning:\nAnswer extracted from completed fi...


In [31]:
print_ablation_wrong_rows(
    full_ablation,
    category="calculus",
    strategy_name="baseline3_adaptive_rules",
    max_rows=30,
    max_question_chars=1200,
    max_response_chars=2500,
)

WRONG ROW 1
id: 1
category: calculus
route_name: calculus_mcq
template_name: baseline3_adaptive_rules_calculus_mcq
is_mcq: True
options: ['$0$', '$frac{1}{a}$', '$frac{3}{a}$', '$frac{1}{2a^2}$', '$frac{1}{2a}$', '$frac{2}{a}$', '$2a$', '$frac{3}{2a}$', '$frac{3}{2a^2}$', '$frac{1}{a^2}$']
gold: F
boxed_answer: E
extracted_final_answer: E
correct: False
schema_valid: True
schema_errors: []
harness_pass_rate: 0.7241379310344828
harness_check_results: [{'name': 'schema_valid', 'passed': True, 'weight': 1.0, 'details': {}}, {'name': 'extractable', 'passed': True, 'weight': 1.0, 'details': {}}, {'name': 'answer_count_valid', 'passed': True, 'weight': 1.0, 'details': {'expected_answer_count': 1, 'actual_answer_count': 1}}, {'name': 'mcq_letter_valid', 'passed': True, 'weight': 1.0, 'details': {'boxed_answer': 'E', 'valid_letters': ['A', 'B', 'C', 'D', 'E', 'F', 'G', 'H', 'I', 'J']}}, {'name': 'calculus_method_evidence', 'passed': None, 'weight': 0.75, 'details': {}}, {'name': 'no_unmapped_n

Each category file under category_work should contain:

1. category name
2. strategy names the teammate is testing
3. harness definition
4. derived rules
5. notes on where prompt templates / strategy registry entries live

## Running full problem set

In [ ]:
baseline3_prompt_chain = build_prompt_chain(strategy_name="baseline3")
baseline3_prompt_rows = []

for problem in baseline3_tagged_val_smoke.problems()[:5]:
    spec = baseline3_prompt_chain.build_spec(problem)
    baseline3_prompt_rows.append({
        "id": problem.id,
        "template_name": spec.name,
        "route_name": spec.metadata.get("route_name"),
        "category": spec.metadata.get("category"),
        "qwen_categories": spec.metadata.get("qwen_categories"),
        "answer_format": spec.metadata.get("answer_format"),
    })

pd.DataFrame(baseline3_prompt_rows)

In [ ]:
# Baseline 3 validation smoke run using Qwen category tags.

baseline3_val_smoke_result = run_baseline3_problem_set(
    problem_set=baseline3_tagged_val_smoke,
    model_bundle=model_bundle,
    generation_config=baseline3_generation_config,
    retry_generation_config=baseline3_retry_generation_config,
    batch_size=BATCH_SIZE,
    limit=None,
    score=True,
    output_jsonl_path=BASELINE3_DIR / "val_smoke_results.jsonl",
    debug_jsonl_path=BASELINE3_DIR / "val_smoke_debug.jsonl",
    report_json_path=BASELINE3_DIR / "val_smoke_report.json",
    comparison_csv_path=EXPERIMENT_COMPARISON_CSV,
    experiment_name="baseline3_qwen_category_prompts_smoke",
    split_name="val_smoke",
    show_progress=True,
)

pprint({
    "summary": baseline3_val_smoke_result.report["summary"],
    "formatting": baseline3_val_smoke_result.report["formatting"],
    "category_counts": baseline3_val_smoke_result.report["category_counts"],
    "comparison_csv": str(EXPERIMENT_COMPARISON_CSV),
})

In [ ]:
def tag_split_for_baseline3(problem_set, split_name, limit=None, reuse_existing=True):
    tag_path = BASELINE3_DIR / f"{split_name}_category_tags.jsonl"
    existing_tags_path = tag_path if reuse_existing and tag_path.exists() else None

    tagged_set, tag_rows = tag_problem_set_with_qwen(
        problem_set=problem_set,
        model_bundle=model_bundle,
        generation_config=category_tag_generation_config,
        batch_size=BATCH_SIZE,
        output_jsonl_path=tag_path,
        existing_tags_path=existing_tags_path,
        limit=limit,
        show_progress=True,
    )

    print(split_name, "tag rows:", len(tag_rows), "path:", tag_path)
    return tagged_set, tag_rows, tag_path


def run_baseline3_split(problem_set, split_name, score, submission=False, limit=None):
    tagged_set, tag_rows, tag_path = tag_split_for_baseline3(
        problem_set=problem_set,
        split_name=split_name,
        limit=limit,
        reuse_existing=True,
    )

    report_path = BASELINE3_DIR / f"{split_name}_report.json"
    result = run_baseline3_problem_set(
        problem_set=tagged_set,
        model_bundle=model_bundle,
        generation_config=baseline3_generation_config,
        retry_generation_config=baseline3_retry_generation_config,
        batch_size=BATCH_SIZE,
        limit=None,
        score=score,
        output_jsonl_path=BASELINE3_DIR / f"{split_name}_results.jsonl",
        debug_jsonl_path=BASELINE3_DIR / f"{split_name}_debug.jsonl",
        submission_csv_path=BASELINE3_DIR / "submission.csv" if submission else None,
        report_json_path=report_path,
        comparison_csv_path=EXPERIMENT_COMPARISON_CSV,
        experiment_name="baseline3_qwen_category_prompts",
        split_name=split_name,
        show_progress=True,
    )

    result.report["category_tags_path"] = str(tag_path)
    result.report["category_tag_count"] = len(tag_rows)
    write_report(result.report, report_path)
    return result

In [ ]:
baseline3_val_result = run_baseline3_split(
    problem_set=val_set,
    split_name="val",
    score=True,
    submission=False,
    limit=None,
)
pprint(baseline3_val_result.report)

# Baseline 4: Supervised Fine-Tuning (SFT)

## Post-SFT Tuning

# Baseline 5: RL